## GeoPoll Questionnaire Validator
Validates a country questionnaire against a reference (template or previous round).
Built with Polars for speed. Supports GeoPoll format (Kobo to be added later).##

In [1]:
import re
import yaml
import polars as pl
import polars.selectors as cs
import openpyxl
from pathlib import Path
from dataclasses import dataclass
from typing import Literal, Optional




## Step 1  Configuration
Set your file paths and validation mode here. This is the only cell you need to edit between runs.
`reference_mode` options:
- `"latest_template"`  compare against the most recent official template in your repo
- `"custom_reference"`  compare against any file you specify
- `"previous_round"`  compare against the previous round questionnaire

In [2]:
ReferenceMode = Literal["latest_template", "custom_reference", "previous_round"]
Enumerator    = Literal["geopoll", "kobo"]
Language      = Literal["EN", "FR", "ES", "AR", "PT"]

@dataclass
class ValidationConfig:
    template_repo          : Path
    working_dir            : Path
    questionnaire_file     : str
    reference_mode         : ReferenceMode
    enumerator             : Enumerator
    language               : Language
    custom_reference_file  : Optional[str]  = None
    previous_round_file    : Optional[str]  = None
    output_subfolder       : str            = "output"
    treat_blank_as_no      : bool           = True   # treat blank Mandatory cell as "no"
    critical_sets_file     : Optional[Path] = None    # path to critical_sets.yaml; None = auto-detect

cfg = ValidationConfig(
    template_repo        = Path(r"C:\Users\edoar\Food and Agriculture Organization\OER - 01. DIEM Monitoring\03. Toolbox\02. Questionnaires"),
    working_dir          = Path(r"C:\Questionnaire_Validation"),
    questionnaire_file   = "GeoPoll_FAO_FR_DRC_R11_test.xlsx",
    reference_mode       = "latest_template",
    enumerator           = "geopoll",
    language             = "FR",
    custom_reference_file = None,
    previous_round_file   = None,
)




## Step 2  Reference file resolution
Finds the correct template file automatically based on enumerator and language,
or uses the file you specified in the config above.

In [3]:
# Maps language code (cfg.language) to the canonical column name in the survey sheet
LANGUAGE_COLUMN: dict[str, str] = {
    "EN": "English",
    "FR": "French",
    "ES": "Spanish",
    "AR": "Arabic",
    "PT": "Portuguese",
}

# Extra aliases used to resolve non-standard headers (e.g. fr_French, sw_Swahili)
LANGUAGE_ALIASES: dict[str, list[str]] = {
    "EN": ["english", "eng", "en"],
    "FR": ["french", "francais", "fran�ais", "fr"],
    "ES": ["spanish", "espanol", "espa�ol", "es"],
    "AR": ["arabic", "arabe", "ar"],
    "PT": ["portuguese", "portugues", "portugu�s", "pt"],
}


def _normalize_header_name(name: str) -> str:
    s = str(name or "").strip().lower()
    return re.sub(r"[^a-z0-9]+", "", s)


def _header_tokens(name: str) -> list[str]:
    s = str(name or "").strip().lower()
    return [t for t in re.split(r"[^a-z0-9]+", s) if t]


def _is_language_candidate_header(name: str) -> bool:
    """Reject obvious non-language columns to avoid false alias matches."""
    n = _normalize_header_name(name)
    blocked = {
        "qname", "suggestedqname", "qtype", "mandatory", "excelrow", "sourcefile",
        "length", "len", "codes", "additionalnotes",
        "defaultskippatternsconditional", "specifyskippatternvariablefrombluetext",
        "specifyskippatternvariable",
    }
    return n not in blocked


def resolve_language_column(columns: list[str], language: str) -> str | None:
    """
    Resolves the best matching text column for a language, supporting variants
    such as 'fr_French' while remaining deterministic.
    """
    lang = str(language or "EN").upper()
    preferred = LANGUAGE_COLUMN.get(lang, "English")
    if preferred in columns:
        return preferred

    preferred_norm = _normalize_header_name(preferred)
    for c in columns:
        if _normalize_header_name(c) == preferred_norm:
            return c

    aliases = [_normalize_header_name(a) for a in LANGUAGE_ALIASES.get(lang, [])]
    ranked: list[tuple[int, int, str]] = []
    for c in columns:
        if not _is_language_candidate_header(c):
            continue
        c_norm = _normalize_header_name(c)
        c_toks = set(_header_tokens(c))
        score = 0

        if preferred_norm and preferred_norm in c_norm:
            score += 6

        for a in aliases:
            if not a:
                continue
            if len(a) <= 2:
                # short language codes (fr/es/ar/pt/en): require token/prefix match,
                # never generic substring match (prevents 'Codes' -> 'Spanish').
                c_low = str(c or "").strip().lower()
                if a in c_toks:
                    score += 5
                if c_low.startswith(a + "_") or c_low.startswith(a + "-"):
                    score += 4
            else:
                if a in c_norm:
                    score += 3
                if a in c_toks:
                    score += 2

        if score > 0:
            ranked.append((score, -len(c), c))

    if not ranked:
        return None
    ranked.sort(reverse=True)
    return ranked[0][2]


def extract_date_from_name(path: Path) -> int:
    """
    Pulls the 8-digit date out of a filename like:
      household_questionnaire_geopoll_EN_template_20250708_ISO3.xlsx
                                                    ^^^^^^^^
    Returns 0 if no date is found, so the file sorts last.
    """
    m = re.search(r"template_(\d{8})", path.name)
    return int(m.group(1)) if m else 0


def find_latest_template(template_repo: Path, enumerator: str, language: str) -> Path:
    """
    Scans the template repo folder and returns the most recent template
    matching the given enumerator (geopoll/kobo) and language (EN/FR/etc.).
    Sorts by date in filename first, then by file modification time as fallback.
    """
    matches = [
        p for p in template_repo.glob("*.xlsx")
        if enumerator.lower() in p.name.lower()
        and f"_{language.lower()}_" in p.name.lower()
        and "template" in p.name.lower()
    ]
    if not matches:
        raise FileNotFoundError(
            f"No template found for enumerator='{enumerator}', language='{language}' in:\n  {template_repo}"
        )
    matches.sort(key=lambda p: (extract_date_from_name(p), p.stat().st_mtime), reverse=True)
    return matches[0]


def resolve_reference_file(cfg: ValidationConfig) -> Path:
    """Returns the path to the reference file based on reference_mode in the config."""
    if cfg.reference_mode == "latest_template":
        return find_latest_template(cfg.template_repo, cfg.enumerator, cfg.language)
    if cfg.reference_mode == "custom_reference":
        if not cfg.custom_reference_file:
            raise ValueError("custom_reference_file must be set when reference_mode='custom_reference'")
        return cfg.working_dir / cfg.custom_reference_file
    if cfg.reference_mode == "previous_round":
        if not cfg.previous_round_file:
            raise ValueError("previous_round_file must be set when reference_mode='previous_round'")
        return cfg.working_dir / cfg.previous_round_file
    raise ValueError(f"Unknown reference_mode: '{cfg.reference_mode}'")


def load_critical_sets(cfg: ValidationConfig) -> dict:
    """
    Loads validation rules from critical_sets.yaml.
    Search order:
      1. cfg.critical_sets_file (if set)
      2. critical_sets.yaml in the current working directory
      3. critical_sets.yaml in a scripts/ subfolder of the current directory

    Returns a dict with keys: exact_sets, min_count_sets, crop_harvest.
    If the file is not found, structural validation is disabled (empty rules returned).
    """
    candidates = [
        Path(cfg.critical_sets_file) if cfg.critical_sets_file else None,
        Path.cwd() / "critical_sets.yaml",
        Path.cwd() / "scripts" / "critical_sets.yaml",
    ]
    for path in candidates:
        if path is not None and path.exists():
            with open(path, "r", encoding="utf-8") as f:
                data = yaml.safe_load(f)
            print(f"Rules loaded  : {path}")
            return {
                "exact_sets"    : data.get("exact_sets", {}),
                "min_count_sets": data.get("min_count_sets", {}),
                "crop_harvest"  : data.get("crop_harvest", {}),
            }
    print("  critical_sets.yaml not found  structural validation disabled.")
    print(f"   Searched: {[str(p) for p in candidates if p is not None]}")
    return {"exact_sets": {}, "min_count_sets": {}, "crop_harvest": {}}


def prepare_run(cfg: ValidationConfig) -> dict:
    """
    Creates output directories and resolves all file paths.
    Returns a dict with the three paths the rest of the notebook needs.
    Prints a summary so you can confirm the right files were picked up.
    """
    cfg.working_dir.mkdir(parents=True, exist_ok=True)
    output_dir = cfg.working_dir / cfg.output_subfolder
    output_dir.mkdir(parents=True, exist_ok=True)

    questionnaire_path = cfg.working_dir / cfg.questionnaire_file
    reference_path     = resolve_reference_file(cfg)

    if not questionnaire_path.exists():
        raise FileNotFoundError(f"Questionnaire file not found:\n  {questionnaire_path}")
    if not reference_path.exists():
        raise FileNotFoundError(f"Reference file not found:\n  {reference_path}")

    report_file = output_dir / f"{questionnaire_path.stem.split()[0]}_{cfg.enumerator}_validation_report.xlsx"
    validated_questionnaire_file = output_dir / f"{questionnaire_path.stem.split()[0]}_{cfg.enumerator}_validated_questionnaire.xlsx"

    run = {
        "questionnaire_path" : questionnaire_path,
        "reference_path"     : reference_path,
        "report_file"        : report_file,
        "validated_questionnaire_file": validated_questionnaire_file,
    }

    print("Questionnaire :", questionnaire_path)
    print("Reference     :", reference_path)
    print("Report output :", report_file)
    print("Validated out :", validated_questionnaire_file)
    return run


run   = prepare_run(cfg)
rules = load_critical_sets(cfg)

# Option comparison target language code (actual columns resolved after loading files)
target_lang = cfg.language.upper()
print(f"Language      : {target_lang}")




Questionnaire : C:\Questionnaire_Validation\GeoPoll_FAO_FR_DRC_R11_test.xlsx
Reference     : C:\Users\edoar\Food and Agriculture Organization\OER - 01. DIEM Monitoring\03. Toolbox\02. Questionnaires\household_questionnaire_geopoll_FR_template_20250708_ISO3.xlsx
Report output : C:\Questionnaire_Validation\output\GeoPoll_FAO_FR_DRC_R11_test_geopoll_validation_report.xlsx
Validated out : C:\Questionnaire_Validation\output\GeoPoll_FAO_FR_DRC_R11_test_geopoll_validated_questionnaire.xlsx
Rules loaded  : c:\Users\edoar\WORK\FAO\repo\questionnaire_validation_revision\scripts\critical_sets.yaml
Language      : FR


## Step 3  Reading and normalising the survey sheet
`read_survey_sheet` loads the Excel file into a Polars DataFrame and tracks the
original Excel row number so every mismatch can be traced back to an exact line.

`build_questions_df` then selects only the columns we care about for validation
and casts everything to clean strings (strip whitespace, fill nulls with `""`).

In [4]:
CORE_COLUMNS = [
    "Q Name",
    "Suggested Qname",
    "English",
    "French",
    "Spanish",
    "Arabic",
    "Portuguese",
    "Q Type",
    "Mandatory",
    "Skip Pattern",
    "Default skip patterns & conditional ",   # note: trailing space is in the real header
    "Specify skip pattern variable (from blue text)",
    "Additional notes",
    "excel_row",
    "source_file",
]


COLUMN_ALIASES: dict[str, list[str]] = {
    "Skip Pattern": [
        "Skip Pattern", "Skip pattern", "Skip Pattern ",
    ],
    "Default skip patterns & conditional ": [
        "Default skip patterns & conditional", "Default skip patterns & conditional ",
        "Default skip patterns and conditional",
    ],
    "Specify skip pattern variable (from blue text)": [
        "Specify skip pattern variable (from blue text)",
        "Specify skip pattern variable",
    ],
}


def _resolve_column_by_aliases(columns: list[str], aliases: list[str]) -> str | None:
    alias_norm = {_normalize_header_name(a) for a in aliases}
    for c in columns:
        if _normalize_header_name(c) in alias_norm:
            return c
    return None


def read_survey_sheet(path: Path, sheet_name: str = "survey", header_row: int = 3, _wb=None) -> pl.DataFrame:
    """
    Reads the survey sheet from an xlsx file into a Polars DataFrame.

    header_row=3 because the GeoPoll template has two title rows above the
    real column headers.

    Two columns are added automatically:
      excel_row    the actual row number in the Excel file (for tracing mismatches)
      source_file  the filename (tells you which file a row came from)

    Pass _wb to reuse an already-opened openpyxl workbook (avoids re-opening
    the same file multiple times).
    """
    owns_wb = _wb is None
    wb = _wb or openpyxl.load_workbook(path, data_only=True, read_only=True)
    try:
        ws = next(
            (wb[n] for n in wb.sheetnames if n.strip().lower() == sheet_name.lower()),
            None
        )
        if ws is None:
            raise KeyError(f"Sheet '{sheet_name}' not found. Available sheets: {wb.sheetnames}")

        row_iter = ws.iter_rows(values_only=True)
        for _ in range(header_row - 1):
            next(row_iter)   # skip the two title rows

        raw_headers = next(row_iter)
        headers = [
            str(h).replace("\n", " ").strip() if h is not None else f"unnamed_{i}"
            for i, h in enumerate(raw_headers, 1)
        ]

        rows      = []
        excel_row = header_row + 1

        for values in row_iter:
            if all(v is None for v in values):
                excel_row += 1
                continue
            row_dict = {headers[i]: values[i] for i in range(len(headers))}
            row_dict["excel_row"]   = excel_row
            row_dict["source_file"] = Path(path).name
            rows.append(row_dict)
            excel_row += 1

        df = pl.DataFrame(rows)
        
        if "Q Name" not in df.columns:
            raise KeyError(f"'Q Name' column not found after reading the sheet. Available columns: {df.columns}")

        df = df.filter(pl.col("Q Name").is_not_null())
   
        return df.with_columns(pl.col("Q Name").cast(pl.Utf8).str.strip_chars())
    finally:
        if owns_wb:
            wb.close()

def build_questions_df(df: pl.DataFrame) -> pl.DataFrame:
    """
    Selects and cleans the core validation columns.
    Warns (does not crash) if an expected column is missing  silent drops
    would make mismatches invisible.
    """
    # Canonicalise language columns first (e.g. fr_French -> French)
    # so downstream logic can rely on stable names.
    mapped: list[tuple[str, str]] = []
    for lang_code, canonical in LANGUAGE_COLUMN.items():
        if canonical in df.columns:
            continue
        matched = resolve_language_column(df.columns, lang_code)
        if matched and matched != canonical:
            df = df.with_columns(pl.col(matched).alias(canonical))
            mapped.append((matched, canonical))
    if mapped:
        print(f"  Language columns mapped: {mapped}")

    # Canonicalise known non-language column variants (especially skip columns).
    mapped_cols: list[tuple[str, str]] = []
    for canonical, aliases in COLUMN_ALIASES.items():
        if canonical in df.columns:
            continue
        matched = _resolve_column_by_aliases(df.columns, aliases)
        if matched and matched != canonical:
            df = df.with_columns(pl.col(matched).alias(canonical))
            mapped_cols.append((matched, canonical))
    if mapped_cols:
        print(f"  Columns mapped: {mapped_cols}")

    missing = [
        c for c in CORE_COLUMNS
        if c not in df.columns and c not in ("excel_row", "source_file")
    ]
    if missing:
        print(f"  Expected columns not found (will be skipped): {missing}")

    available = [c for c in CORE_COLUMNS if c in df.columns]
    out = df.select(available)

    out = out.with_columns(
        cs.exclude("excel_row").cast(pl.Utf8).fill_null("").str.strip_chars()
    )
    return out




## Step 4  Exploding answer options
Instead of comparing the full English text block as one string (old approach),
we parse each question's numbered options into individual rows.

This means the diff can say **"option 3 of `resp_gender` changed from X to Y"**
rather than just **"`resp_gender` English text differs"**  giving you the exact
location of the change.

The two questions with no parseable options (`crp_main`, `crp_salesmain`) are
expected: their options are injected dynamically at run time. They are handled
by the presence/absence checks, not here.

In [ ]:
OPTION_BEARING_TYPES = {
    "Single Choice",
    "Open Ended-Single Choice",
    "Open Ended - Single Choice",
    "Select All That Apply",
    "Open Ended-Select All That Apply",
    "Open Ended - Select All That Apply",
    "Open Ended - Select All That Apply ",   # trailing-space variant seen in real files
}

# Matches blocks like:
#   1) Label text
#   2) Next label
#
# (?m)     ^ matches the start of each line
# DOTALL   . also matches newlines, so multi-line labels are captured in full
OPTION_PATTERN = re.compile(r"(?m)^\s*(\d+)\)\s*(.*?)(?=^\s*\d+\)|\Z)", re.DOTALL)


def parse_options(text: str) -> list[tuple[int, str]]:
    """
    Parses numbered options from a question text cell.
    Returns a list of (option_code, option_label) tuples.

    Labels are whitespace-collapsed but otherwise kept raw 
    

    Example:
        input:  "What is your gender?\\n\\n1)Male\\n2)Female\\n3)DON'T want to answer"
        output: [(1, "Male"), (2, "Female"), (3, "DON'T want to answer")]
    """
    if not text or not isinstance(text, str):
        return []
    results = []
    for m in OPTION_PATTERN.finditer(text):
        code  = int(m.group(1))
        label = " ".join(m.group(2).split())   # collapse newlines/extra spaces
        if label:
            results.append((code, label))
    return results


def explode_options(questions_df: pl.DataFrame, text_col: str = "English") -> pl.DataFrame:
    """
    Turns the questions DataFrame (one row per question) into an options
    DataFrame (one row per answer option).

    Output columns:
        Q Name, Q Type, option_code (Int64), option_label (Utf8),
        excel_row, source_file
    """
    EMPTY_SCHEMA = {
        "Q Name"      : pl.Utf8,
        "Q Type"      : pl.Utf8,
        "option_code" : pl.Int64,
        "option_label": pl.Utf8,
        "excel_row"   : pl.Int64,
        "source_file" : pl.Utf8,
    }

    if text_col not in questions_df.columns:
        print(f"  Column '{text_col}' not found  returning empty options frame.")
        return pl.DataFrame(schema=EMPTY_SCHEMA)

    df = (
        questions_df.filter(pl.col("Q Type").is_in(list(OPTION_BEARING_TYPES)))
        if "Q Type" in questions_df.columns
        else questions_df
    )

    carry_cols = [c for c in ["Q Name", "Q Type", "excel_row", "source_file"] if c in df.columns]
    rows = []

    for row in df.select(carry_cols + [text_col]).iter_rows(named=True):
        text = row.get(text_col) or ""
        for code, label in parse_options(text):
            out_row = {c: row.get(c) for c in carry_cols}
            out_row["option_code"]  = code
            out_row["option_label"] = label
            rows.append(out_row)

    if not rows:
        return pl.DataFrame(schema=EMPTY_SCHEMA)

    return pl.DataFrame(rows).with_columns(
        pl.col("option_code").cast(pl.Int64),
        pl.col("option_label").cast(pl.Utf8),
    )



In [ ]:
PLACEHOLDER_PATTERN = re.compile(r"\$[^$\r\n]+\$")

RESTORE_TEXT_COLUMNS = [
    "English",
    "French",
    "Spanish",
    "Arabic",
    "Portuguese",
    "Default skip patterns & conditional ",
    "Specify skip pattern variable (from blue text)",
    "Additional notes",
]

SEASON_PHASE_EXPECTED_VALUES = {
    "not yet in season",
    "land preparation",
    "planting",
    "early growing",
    "growing",
    "maturing",
    "Harvesting",
    "Recently finished"
}

CROP_MARKER_HINTS = {
    "$TEN MOST COMMON CROPS$",
    "$CROP CODES$",
    "$CROP SOLD CODES$",
}


def _clean_header(value) -> str:
    return str(value or "").replace("\n", " ").strip()


def _find_column(columns: list[str], *candidates: str) -> Optional[str]:
    lookup = {_clean_header(c).lower(): c for c in columns}
    for candidate in candidates:
        match = lookup.get(_clean_header(candidate).lower())
        if match:
            return match
    return None


def read_generic_sheet(path: Path, sheet_name: str, header_row: int, _wb=None) -> pl.DataFrame:
    """
    Reads any workbook sheet into Polars using the provided 1-based header row.
    Used for auxiliary sheets such as Crop list and Additional information.
    """
    owns_wb = _wb is None
    wb = _wb or openpyxl.load_workbook(path, data_only=True, read_only=True)
    try:
        ws = next((wb[n] for n in wb.sheetnames if n.strip().lower() == sheet_name.lower()), None)
        if ws is None:
            raise KeyError(f"Sheet '{sheet_name}' not found in {path.name}. Available sheets: {wb.sheetnames}")

        row_iter = ws.iter_rows(values_only=True)
        for _ in range(header_row - 1):
            next(row_iter)

        raw_headers = next(row_iter)
        headers = [
            _clean_header(h) if h is not None else f"unnamed_{i}"
            for i, h in enumerate(raw_headers, 1)
        ]

        rows = []
        for values in row_iter:
            if all(v is None for v in values):
                continue
            rows.append({headers[i]: values[i] for i in range(len(headers))})

        return pl.DataFrame(rows) if rows else pl.DataFrame(schema={h: pl.Utf8 for h in headers})
    finally:
        if owns_wb:
            wb.close()

def read_additional_information(path: Path, _wb=None) -> pl.DataFrame:
    return read_generic_sheet(path, sheet_name="Additional information", header_row=2, _wb=_wb)


def read_crop_list(path: Path, _wb=None) -> pl.DataFrame:
    return read_generic_sheet(path, sheet_name="Crop list", header_row=3, _wb=_wb)

# Returns True if the value contains at least one $...$ placeholder.
def has_placeholder(value) -> bool:
    return bool(PLACEHOLDER_PATTERN.search(str(value or "")))


def build_placeholder_restore_map(
    reference_questions: pl.DataFrame,
    candidate_columns: Optional[list[str]] = None,
) -> dict[str, dict[str, str]]:
    """
    Returns a per-question map of columns whose template/reference cell still
    contains a $...$ placeholder. Those exact reference cells are what we use
    to restore the current questionnaire before comparison.
    """
    columns = [c for c in (candidate_columns or RESTORE_TEXT_COLUMNS) if c in reference_questions.columns]
    restore_map: dict[str, dict[str, str]] = {}

    for row in reference_questions.select(["Q Name"] + columns).iter_rows(named=True):
        qname = row["Q Name"]
        hits = {
            col: str(row[col] or "")
            for col in columns
            if has_placeholder(row.get(col))
        }
        if hits:
            restore_map[qname] = hits
    return restore_map


def restore_placeholder_cells(
    current_questions: pl.DataFrame,
    reference_questions: pl.DataFrame,
    candidate_columns: Optional[list[str]] = None,
) -> tuple[pl.DataFrame, pl.DataFrame]:
    """
    Restores current questionnaire cells to the template/reference version,
    but only for cells where the reference still contains a $...$ placeholder.
    Returns (restored_questions, restore_log).
    """
    restore_map = build_placeholder_restore_map(reference_questions, candidate_columns)
    if not restore_map:
        empty_log = pl.DataFrame(schema={"Q Name": pl.Utf8, "field": pl.Utf8, "restored_value": pl.Utf8})
        return current_questions.clone(), empty_log

    restore_log_rows = []
    for qname, hits in restore_map.items():
        for field, restored_value in hits.items():
            restore_log_rows.append({
                "Q Name": qname,
                "field": field,
                "restored_value": restored_value,
            })

    if not restore_log_rows:
        empty_log = pl.DataFrame(schema={"Q Name": pl.Utf8, "field": pl.Utf8, "restored_value": pl.Utf8})
        return current_questions.clone(), empty_log

    restore_log = pl.DataFrame(restore_log_rows).select(["Q Name", "field", "restored_value"])
    restore_wide = restore_log.pivot(
        values="restored_value",
        index="Q Name",
        columns="field",
        aggregate_function="first",
    )

    joined = current_questions.join(restore_wide, on="Q Name", how="left", suffix="_restore")

    fields = sorted(set(restore_log["field"].to_list()))
    for field in fields:
        restore_col = f"{field}_restore"
        if field in current_questions.columns and restore_col in joined.columns:
            joined = (
                joined
                .with_columns(pl.coalesce([pl.col(restore_col), pl.col(field)]).alias(field))
                .drop(restore_col)
            )

    leftover_restore_cols = [c for c in joined.columns if c.endswith("_restore")]
    if leftover_restore_cols:
        joined = joined.drop(leftover_restore_cols)

    restored = joined.select(current_questions.columns)
    return restored, restore_log

def build_additional_info_replacements(path: Path, language: str, _wb=None) -> dict[str, str]:
    """
    Builds the forward substitution map from the questionnaire's Additional
    information sheet. Keys must already be written in the sheet as $...$
    placeholders; this keeps the logic data-driven rather than hardcoded.
    """
    df = read_additional_information(path, _wb=_wb)
    if df.height == 0:
        return {}

    replacement_col = _find_column(
        df.columns,
        f"Replacement ({language})",
        "Replacement",
        "Replacement (EN)",
    )
    original_col = _find_column(df.columns, "Original")
    if not replacement_col or not original_col:
        return {}

    replacements: dict[str, str] = {}
    season_phase_value = ""

    for row in df.iter_rows(named=True):
        original = str(row.get(original_col) or "").strip()
        replacement = str(row.get(replacement_col) or "").strip()
        if has_placeholder(original) and replacement:
            replacements[original] = replacement
        if original.lower() in {"$season phase$", "$season phase $"}:
            season_phase_value = replacement.lower()

    if "$expected or nothing$" in replacements:
        replacements["$expected or nothing$"] = (
            "expected" if season_phase_value in SEASON_PHASE_EXPECTED_VALUES else ""
        )

    return replacements

def apply_text_replacements(value, replacements: dict[str, str]) -> str:
    text = str(value or "")
    for placeholder, replacement in replacements.items():
        text = text.replace(placeholder, replacement)
    return text


def apply_placeholder_conversions(
    questions_df: pl.DataFrame,
    replacements: dict[str, str],
    candidate_columns: Optional[list[str]] = None,
) -> pl.DataFrame:
    """
    Forward-applies placeholder replacements to text-bearing columns.
    This is used later when we rebuild the validated deployed questionnaire.
    """
    columns = [c for c in (candidate_columns or RESTORE_TEXT_COLUMNS) if c in questions_df.columns]
    if not replacements or not columns:
        return questions_df.clone()

    out = questions_df.clone()
    for col in columns:
        expr = pl.col(col).cast(pl.Utf8).fill_null("")
        for placeholder, replacement in replacements.items():
            expr = expr.str.replace_all(re.escape(placeholder), replacement)
        out = out.with_columns(expr.alias(col))
    return out

def extract_crop_metadata(crop_df: pl.DataFrame, language: str) -> dict:
    """
    Pulls the key columns from the Crop list sheet in a language-aware way.
    Returned labels include the full crop pool, not only the selected rows.
    """
    label_col = _find_column(crop_df.columns, f"Label ({language})", "Label (EN)")
    dataset_col = _find_column(crop_df.columns, "Dataset code")
    select_col = _find_column(crop_df.columns, "Select top 10 crops", "Select top 10 crops ")

    if not label_col:
        return {"label_col": None, "dataset_col": dataset_col, "select_col": select_col, "labels": []}

    labels = [
        str(v).strip()
        for v in crop_df.get_column(label_col).to_list()
        if str(v or "").strip()
    ]
    return {
        "label_col": label_col,
        "dataset_col": dataset_col,
        "select_col": select_col,
        "labels": labels,
    }


def find_crop_placeholder_questions(reference_questions: pl.DataFrame) -> list[str]:
    """
    Identifies questions whose reference/template text carries a crop marker.
    Those are the questions that need template-style restoration for comparison
    and deployed-style rebuilding for validated output.
    """
    text_columns = [c for c in RESTORE_TEXT_COLUMNS if c in reference_questions.columns]
    hits = []
    for row in reference_questions.select(["Q Name"] + text_columns).iter_rows(named=True):
        values = "\n".join(str(row.get(col) or "") for col in text_columns)
        if any(marker in values for marker in CROP_MARKER_HINTS):
            hits.append(row["Q Name"])
    return sorted(set(hits))


def build_crop_option_block(crop_df: pl.DataFrame, language: str, include_no_crop: bool = True) -> str:
    """
    Builds a numbered crop option block from the Crop list sheet.
    This is the deployed representation that we will later write to the
    validated questionnaire output.
    """
    meta = extract_crop_metadata(crop_df, language)
    label_col = meta["label_col"]
    if not label_col:
        return ""

    labels = [
        str(v).strip()
        for v in crop_df.get_column(label_col).to_list()
        if str(v or "").strip()
    ]
    lines = [f"{idx}){label}" for idx, label in enumerate(labels, start=1)]
    if include_no_crop:
        lines.extend(["91)No crop production", "92)DON'T KNOW", "93)REFUSED"])
    else:
        lines.extend(["92)DON'T KNOW", "93)REFUSED"])
    return "\n".join(lines)


def rebuild_crop_questions_for_deployed_form(
    questions_df: pl.DataFrame,
    template_questions: pl.DataFrame,
    crop_df: pl.DataFrame,
    language: str,
    candidate_columns: Optional[list[str]] = None,
) -> pl.DataFrame:
    """
    Replaces the crop placeholder inside crop-driven questions with the crop
    list block generated from the current questionnaire's Crop list sheet.
    Used in previous-round mode so we compare deployed-vs-deployed.
    """
    crop_qnames = set(find_crop_placeholder_questions(template_questions))
    columns = [c for c in (candidate_columns or RESTORE_TEXT_COLUMNS) if c in questions_df.columns]
    if not crop_qnames or not columns:
        return questions_df.clone()

    main_block = build_crop_option_block(crop_df, language, include_no_crop=True)
    sold_block = build_crop_option_block(crop_df, language, include_no_crop=False)
    if not main_block:
        return questions_df.clone()

    crop_qnames_list = sorted(crop_qnames)
    marker_pattern = re.escape("$TEN MOST COMMON CROPS$")
    qname_lc = pl.col("Q Name").cast(pl.Utf8).str.to_lowercase()
    is_crop_q = pl.col("Q Name").is_in(crop_qnames_list)
    is_sales_q = qname_lc.str.contains("sales")

    exprs = []
    for col in columns:
        base = pl.col(col).cast(pl.Utf8).fill_null("")
        has_marker = base.str.contains(marker_pattern)
        replaced_main = base.str.replace_all(marker_pattern, main_block)
        replaced_sold = base.str.replace_all(marker_pattern, sold_block)
        exprs.append(
            pl.when(is_crop_q & has_marker)
            .then(
                pl.when(is_sales_q).then(replaced_sold).otherwise(replaced_main)
            )
            .otherwise(base)
            .alias(col)
        )

    return questions_df.with_columns(exprs)






In [7]:
current_wb = openpyxl.load_workbook(run["questionnaire_path"], data_only=True, read_only=True)
current_raw = read_survey_sheet(run["questionnaire_path"], _wb=current_wb)
reference_raw = read_survey_sheet(run["reference_path"])

current_questions_raw = build_questions_df(current_raw)
reference_questions_raw = build_questions_df(reference_raw)

current_crop_list = read_crop_list(run["questionnaire_path"], _wb=current_wb)
current_text_replacements = build_additional_info_replacements(run["questionnaire_path"], cfg.language, _wb=current_wb)
current_wb.close()
restore_log                = pl.DataFrame(schema={"Q Name": pl.Utf8, "field": pl.Utf8, "restored_value": pl.Utf8})
template_questions_for_restore = reference_questions_raw

if cfg.reference_mode == "previous_round":
    template_path = find_latest_template(cfg.template_repo, cfg.enumerator, cfg.language)
    template_raw  = read_survey_sheet(template_path)
    template_questions_for_restore = build_questions_df(template_raw)

    restored_current_questions, restore_log = restore_placeholder_cells(
        current_questions_raw,
        template_questions_for_restore,
    )
    comparison_current_questions = apply_placeholder_conversions(
        restored_current_questions,
        current_text_replacements,
    )
    comparison_current_questions = rebuild_crop_questions_for_deployed_form(
        comparison_current_questions,
        template_questions_for_restore,
        current_crop_list,
        cfg.language,
    )
    comparison_reference_questions = reference_questions_raw
    validated_output_questions = comparison_current_questions
else:
    comparison_current_questions, restore_log = restore_placeholder_cells(
        current_questions_raw,
        reference_questions_raw,
    )
    comparison_reference_questions = reference_questions_raw
    validated_output_questions = apply_placeholder_conversions(
        current_questions_raw,
        current_text_replacements,
    )
    validated_output_questions = rebuild_crop_questions_for_deployed_form(
        validated_output_questions,
        reference_questions_raw,
        current_crop_list,
        cfg.language,
    )

current_questions   = comparison_current_questions
reference_questions = comparison_reference_questions

current_en_col   = resolve_language_column(current_questions.columns, "EN")
reference_en_col = resolve_language_column(reference_questions.columns, "EN")
if not current_en_col or not reference_en_col:
    raise KeyError("English column not found in one of the files; option checks require English baseline.")

current_tgt_col   = resolve_language_column(current_questions.columns, target_lang)
reference_tgt_col = resolve_language_column(reference_questions.columns, target_lang)

if target_lang == "EN":
    current_tgt_col   = current_en_col
    reference_tgt_col = reference_en_col

if not current_tgt_col or not reference_tgt_col:
    print(f"  Target language column unresolved for {target_lang}; falling back to English for option label comparison.")
    current_tgt_col   = current_en_col
    reference_tgt_col = reference_en_col

print(f"  Option columns -> EN: current='{current_en_col}', reference='{reference_en_col}'")
print(f"  Option columns -> {target_lang}: current='{current_tgt_col}', reference='{reference_tgt_col}'")

# English options are the stable baseline for presence (added/removed option codes)
current_options_en   = explode_options(current_questions, text_col=current_en_col)
reference_options_en = explode_options(reference_questions, text_col=reference_en_col)

# Target-language options are used for label mismatch checks.
# If target columns resolve to EN, reuse EN exploded frames to avoid duplicate work.
if current_tgt_col == current_en_col and reference_tgt_col == reference_en_col:
    current_options_tgt   = current_options_en.clone()
    reference_options_tgt = reference_options_en.clone()
else:
    current_options_tgt   = explode_options(current_questions, text_col=current_tgt_col)
    reference_options_tgt = explode_options(reference_questions, text_col=reference_tgt_col)

print("--- comparison current questionnaire ---")
print(f"  questions : {current_questions.shape}")
print(f"  options EN      : {current_options_en.shape}")
print(f"  options {target_lang:<2}      : {current_options_tgt.shape}")
print(f"  restored placeholder cells : {restore_log.height}")
print(f"  validated output questions : {validated_output_questions.shape}")

print("\n--- comparison reference file ---")
print(f"  questions : {reference_questions.shape}")
print(f"  options EN      : {reference_options_en.shape}")
print(f"  options {target_lang:<2}      : {reference_options_tgt.shape}")

print("\n--- sample: resp_gender options (comparison current, target language) ---")
print(current_options_tgt.filter(pl.col("Q Name") == "resp_gender"))





  Language columns mapped: [('fr_French', 'French')]
  Columns mapped: [('Default skip patterns & conditional', 'Default skip patterns & conditional '), ('Specify skip pattern variable', 'Specify skip pattern variable (from blue text)')]
  Expected columns not found (will be skipped): ['Spanish', 'Arabic', 'Portuguese', 'Additional notes']
  Columns mapped: [('Default skip patterns & conditional', 'Default skip patterns & conditional ')]
  Expected columns not found (will be skipped): ['Spanish', 'Arabic', 'Portuguese', 'Additional notes']
  Option columns -> EN: current='English', reference='English'
  Option columns -> FR: current='French', reference='French'
--- comparison current questionnaire ---
  questions : (196, 11)
  options EN      : (906, 6)
  options FR      : (909, 6)
  restored placeholder cells : 52
  validated output questions : (196, 11)

--- comparison reference file ---
  questions : (180, 11)
  options EN      : (845, 6)
  options FR      : (844, 6)

--- sample: re

C:\Users\edoar\AppData\Local\Temp\ipykernel_40652\4082875250.py:140: DeprecationWarning: the argument `columns` for `DataFrame.pivot` is deprecated. It was renamed to `on` in version 1.0.0.
  restore_wide = restore_log.pivot(


## Step 6  Normalisation helpers
Two helpers used by the comparison functions below:

- `normalize_text_expr`  lowercases, removes punctuation, collapses spaces.
  Used for option label comparison so "DON'T KNOW" and "Don't know" are treated as equal.
- `normalize_mandatory_expr`  maps any variant of yes/no/true/false/1/0 to a
  canonical `"yes"` or `"no"` so formatting differences don't create false mismatches.

In [8]:
def normalize_text_expr(col_name: str) -> pl.Expr:
    """
    Polars expression that lowercases a string column, removes punctuation,
    and collapses whitespace. Apply as an alias so the original column is preserved.

    Usage:
        df.with_columns(normalize_text_expr("option_label").alias("option_label_norm"))
    """
    return (
        pl.col(col_name)
        .cast(pl.Utf8)
        .fill_null("")
        .str.to_lowercase()
        .str.replace_all(r"[^\w\s]", "")   # remove punctuation
        .str.replace_all(r"\s+", " ")       # collapse multiple spaces
        .str.strip_chars()
    )


def normalize_mandatory_expr(col_name: str) -> pl.Expr:
    """
    Polars expression that maps any common yes/no variant to canonical "yes"/"no"/""
    so that "Yes", "YES", "y", "true", "1" are all treated as equivalent.

    Using when/then/otherwise keeps this inside the Polars execution engine
    (faster and parallelisable) vs a Python lambda with map_elements.
    """
    cleaned = (
        pl.col(col_name)
        .cast(pl.Utf8)
        .fill_null("")
        .str.to_lowercase()
        .str.strip_chars()
    )
    return (
        pl.when(cleaned.is_in(["yes", "y", "true", "1"])).then(pl.lit("yes"))
        .when(cleaned.is_in(["no",  "n", "false", "0"])).then(pl.lit("no"))
        .otherwise(pl.lit(""))
    )



## Step 7  Comparison and validation functions
Seven focused functions, each returning a tidy DataFrame in the common issues schema:

| Function | What it checks | Source of rules |
|---|---|---|
| `compare_question_presence` | Questions added or removed vs reference; `o_*` optional questions tagged as *info* | reference file |
| `compare_mandatory` | Mandatory flag changed for any shared question (`treat_blank_as_no` via config) | reference file |
| `compare_option_labels` | Individual answer option text changed  language-aware (`cfg.language`) | reference file |
| `validate_critical_sets` | Specific named questions must be present with correct Mandatory flag | `critical_sets.yaml`  `exact_sets` |
| `validate_prefix_counts` | Groups must meet minimum counts: CS coping strategies (4/3/3), HDDS (3) | `critical_sets.yaml`  `min_count_sets` |
| `validate_crop_harvest` | Crop harvest must be either the minimal set or the full set  no partial | `critical_sets.yaml`  `crop_harvest` |
| `validate_skip_patterns` | Questions referenced in skip patterns must still exist  rules derived from reference, no hardcoding | reference file |


In [9]:
SKIP_PATTERN_COLS = [
    "Skip Pattern",
    "Default skip patterns & conditional ",   # trailing space is intentional
    "Specify skip pattern variable (from blue text)",
]


def _normalize_skip_text(value) -> str:
    text = "" if value is None else str(value)
    return re.sub(r"\s+", " ", text).strip()


def _parse_skip_option_codes(
    skip_text: str,
    max_range_span: int = 200,
    max_total_codes: int = 500,
) -> set[int]:
    """
    Extracts numeric option codes from the left side of skip rules.
    Examples handled: "1=...", "1-4 = ...", "1,3,5=...", "2 to 4 = ...".

    Safety guards:
      - do not fully expand very wide ranges,
      - cap total extracted codes to avoid pathological blowups.
    """
    if not skip_text:
        return set()

    codes = set()
    parts = [p.strip() for p in re.split(r"[\r\n;]+", str(skip_text)) if p.strip()]
    for part in parts:
        if len(codes) >= max_total_codes:
            break

        left = part.split("=", 1)[0] if "=" in part else part

        for a, b in re.findall(r"\b(\d+)\s*(?:-|to)\s*(\d+)\b", left, flags=re.IGNORECASE):
            start, end = int(a), int(b)
            lo, hi = sorted((start, end))
            span = hi - lo + 1

            if span <= max_range_span and (len(codes) + span) <= max_total_codes:
                codes.update(range(lo, hi + 1))
            else:
                # Keep endpoints so we still validate something without exploding runtime.
                codes.add(lo)
                codes.add(hi)
                if len(codes) >= max_total_codes:
                    break

        if len(codes) >= max_total_codes:
            break

        left_no_ranges = re.sub(r"\b\d+\s*(?:-|to)\s*\d+\b", " ", left, flags=re.IGNORECASE)
        for n in re.findall(r"\b\d+\b", left_no_ranges):
            codes.add(int(n))
            if len(codes) >= max_total_codes:
                break

    return codes

def _extract_referenced_qnames(skip_text: str, source_q: str, ref_qnames: set[str], curr_qnames: set[str]) -> set[str]:
    """
    Extract target question names from skip rules using a strict heuristic:
      - parse only RHS of '=' (or line/segment after '='),
      - take the first token as candidate target,
      - ignore common routing labels (ineligible, end, quota, etc.),
      - keep candidates that are known qnames OR look like qnames (underscore).
    """
    text = str(skip_text or "")
    if len(text) > 4000:
        text = text[:4000]

    stopwords = {
        "ineligible", "eligible", "end", "poll", "quota", "reached",
        "terminate", "terminated", "screenout", "screen_out",
        "refusal", "refused", "closeout", "close", "callback",
        "yes", "no", "na", "n_a", "none",
    }

    out = set()
    segments = [s.strip() for s in re.split(r"[\r\n;]+", text) if s.strip()]
    for seg in segments:
        if "=" not in seg:
            continue
        rhs = seg.split("=", 1)[1].strip()
        if not rhs:
            continue

        m = re.match(r"^\s*([A-Za-z_]\w*)", rhs)
        if not m:
            continue

        cand = m.group(1)
        cand_l = cand.lower()
        if cand == source_q or cand_l in stopwords:
            continue

        if cand in ref_qnames or cand in curr_qnames or ("_" in cand):
            out.add(cand)

    return out


# ─── helpers for Default-column consistency check ────────────────────────────

_QNAME_TOKEN_RE = re.compile(r'\b([a-z][a-z0-9_]{2,})\b')
_GO_TO_RE       = re.compile(r'\bgo\s+to\s+([^\n;]+)', re.IGNORECASE)
_NEXT_Q_RE      = re.compile(r'\bnext\s+question\b', re.IGNORECASE)
_OPTIONAL_RE    = re.compile(r'\boptional\s+question\b', re.IGNORECASE)


def _extract_qname_tokens(text: str, known_qnames: set[str]) -> list[str]:
    """Return word tokens from text that are known Q Names."""
    return [t for t in _QNAME_TOKEN_RE.findall(str(text or "")) if t in known_qnames]


def _extract_qname_like(text: str, known_qnames: set[str]) -> list[str]:
    """
    Return word tokens that are either known Q Names OR look like Q Names
    (contain an underscore).  This lets us detect references to Q Names
    that were renamed / removed from the current questionnaire.
    """
    return [t for t in _QNAME_TOKEN_RE.findall(str(text or ""))
            if t in known_qnames or "_" in t]


def _extract_condition_codes(stmt: str) -> set[int]:
    """
    Extract option codes from a full Default-column rule statement by finding
    the range between '=' and 'go to', e.g.:
      "if hh_gender = 1-4, go to hh_education"  →  {1, 2, 3, 4}
      "For any response, go to X"               →  {} (any response = no range)
    Receives the FULL statement (not sliced) so that 'go to' can anchor the match.
    """
    match = re.search(
        r'=\s*([\d][^;=\n]*?)\s*,?\s*go\s+to',
        stmt, re.IGNORECASE,
    )
    if not match:
        return set()
    rng = match.group(1).strip()
    codes: set[int] = set()
    for a, b in re.findall(r'\b(\d+)\s*(?:-|to)\s*(\d+)\b', rng, re.IGNORECASE):
        codes.update(range(*sorted((int(a), int(b) + 1))))
    clean = re.sub(r'\b\d+\s*(?:-|to)\s*\d+\b', ' ', rng, flags=re.IGNORECASE)
    codes.update(int(n) for n in re.findall(r'\b\d+\b', clean))
    return codes


def _extract_skip_codes_for_target(skip_text: str, target: str) -> set[int]:
    """
    Extract option codes from Skip-Pattern lines that send flow to *target*,
    e.g. "1-3 = hh_education"  →  {1, 2, 3}.
    """
    codes: set[int] = set()
    for part in re.split(r'[\r\n;]+', str(skip_text or "")):
        if target not in part or "=" not in part:
            continue
        left = part.split("=", 1)[0]
        for a, b in re.findall(r'\b(\d+)\s*(?:-|to)\s*(\d+)\b', left, re.IGNORECASE):
            codes.update(range(*sorted((int(a), int(b) + 1))))
        clean = re.sub(r'\b\d+\s*(?:-|to)\s*\d+\b', ' ', left, flags=re.IGNORECASE)
        codes.update(int(n) for n in re.findall(r'\b\d+\b', clean))
    return codes



def _parse_default_skip_rules(default_text: str, known_qnames: set[str]) -> list[dict]:
    """
    Parse "Default skip patterns & conditional" into structured rules.
    Each rule: {targets, is_flexible, is_next_question, raw}

    Examples handled:
      "if X = 1-8, go to ls_proddif"
      "if X = 2-4, go to [ls_salesmain OR optional question]"
      "For any response, go to ls_proddif"
      "if X = 1, go to [next question]"
      "if X = 1-11, go to ls_salesdif; if X = 12-13, go to fish_intro"
    """
    text = _normalize_skip_text(str(default_text or ""))
    if not text:
        return []

    rules = []
    for stmt in re.split(r'[;\n]+', text):
        stmt = stmt.strip()
        if not stmt:
            continue

        go_match = _GO_TO_RE.search(stmt)
        if not go_match:
            continue

        target_text      = go_match.group(1).strip()
        is_next_question = bool(_NEXT_Q_RE.search(target_text))
        is_flexible      = bool(_OPTIONAL_RE.search(target_text))
        targets          = _extract_qname_like(target_text, known_qnames)

        # Extract option codes from the full statement (regex anchors on "go to")
        option_codes = _extract_condition_codes(stmt)

        # Only record rule if there is something to check
        if targets or is_next_question:
            rules.append({
                "targets"         : targets,
                "is_flexible"     : is_flexible,
                "is_next_question": is_next_question,
                "option_codes"    : option_codes,
                "raw"             : stmt,
            })
    return rules


def _check_skip_consistency(
    skip_text      : str,
    rules          : list[dict],
    q_mandatory_map: dict[str, str],
    curr_qnames    : set[str],
    next_qname     : str | None,
) -> list[str]:
    """
    Returns a list of issue descriptions (empty = consistent).

    Rules:
    - Each required target Q Name must appear in skip_text.
    - If the rule is flexible ("OR optional question"), a non-mandatory Q Name
      in skip_text is also acceptable.
    - Rules whose only target is [next question] are skipped (too contextual).
    """
    skip_present = set(_extract_qname_like(str(skip_text or ""), curr_qnames))
    issues = []

    for rule in rules:
        targets          = rule["targets"]
        is_flexible      = rule["is_flexible"]
        is_next_question = rule["is_next_question"]

        # [next question] with no concrete target → too contextual, do not flag
        if is_next_question and not targets:
            continue
        if not targets:
            continue

        # Happy path: at least one required target is present in the skip pattern
        found_targets = [t for t in targets if t in skip_present]
        if found_targets:
            # Also verify the option range matches (if the Default rule specifies one)
            expected_codes = rule.get("option_codes", set())
            if expected_codes:
                for ft in found_targets:
                    actual_codes = _extract_skip_codes_for_target(str(skip_text), ft)
                    if actual_codes and actual_codes != expected_codes:
                        issues.append(
                            f"option range mismatch for target '{ft}': "
                            f"default says {sorted(expected_codes)}, "
                            f"skip pattern has {sorted(actual_codes)}"
                        )
            continue

        if is_flexible:
            # Flexible rule: a non-mandatory alternative is also acceptable
            non_mand = [q for q in skip_present
                        if q_mandatory_map.get(q) not in ("mandatory", "mandatory-panel")]
            if non_mand:
                continue
            issues.append(
                f"expected {targets} (or non-mandatory alternative) not found in skip pattern"
            )
        else:
            if not skip_text.strip():
                issues.append(
                    f"skip pattern is empty but default column says go to {targets}"
                )
            else:
                issues.append(
                    f"expected {targets} in skip pattern; got: {skip_text[:80]}"
                )

    return issues


# ─── main validation function ────────────────────────────────────────────────

def validate_skip_patterns(
    current_questions    : pl.DataFrame,
    reference_questions  : pl.DataFrame,
    current_options_en   : pl.DataFrame | None = None,
    reference_options_en : pl.DataFrame | None = None,
    option_issue_qnames  : set[str] | None = None,
    q_mandatory_map      : dict[str, str] | None = None,
) -> pl.DataFrame:
    """
    Three-layer skip-pattern validation:

    1. Consistency check (high): Skip Pattern must match the logic described in
       "Default skip patterns & conditional" in the CURRENT file.  This replaces
       the old ref-vs-current structural diff, which produced false positives
       because country teams are expected to fill in bracket placeholders.

    2. Default-column drift (medium): flags if the "Default skip patterns &
       conditional" column itself changed between reference and current.  That
       column is the authoritative spec; a change there is worth reviewing.

    3. Option code validity (high): numeric codes referenced in Skip Pattern
       must actually exist in the question's answer options.

    4. Option drift risk (medium): a question that has skip logic also had its
       option set changed — the skip ranges may need updating.
    """
    EMPTY_SCHEMA = {
        "issue_type": pl.Utf8, "set_name": pl.Utf8, "Q Name": pl.Utf8,
        "field"     : pl.Utf8, "current" : pl.Utf8, "reference": pl.Utf8,
        "severity"  : pl.Utf8, "excel_row": pl.Int64,
    }

    DEF_COL  = "Default skip patterns & conditional "
    SKIP_COL = "Skip Pattern"

    has_def_cur  = DEF_COL  in current_questions.columns
    has_skip_cur = SKIP_COL in current_questions.columns
    has_def_ref  = DEF_COL  in reference_questions.columns

    if not has_def_cur and not has_skip_cur:
        return pl.DataFrame(schema=EMPTY_SCHEMA)

    curr_qnames         = set(current_questions["Q Name"].to_list())
    q_mandatory_map     = q_mandatory_map or {}
    option_issue_qnames = option_issue_qnames or set()

    # Option code lookup for validity check (layer 3)
    curr_opt_map: dict[str, set[int]] = {}
    if current_options_en is not None and current_options_en.height > 0:
        for row in current_options_en.select(["Q Name", "option_code"]).iter_rows(named=True):
            q, c = row.get("Q Name"), row.get("option_code")
            if q and c is not None:
                curr_opt_map.setdefault(q, set()).add(int(c))

    # Minimal row maps — only the three columns we need
    cur_cols = ["Q Name"] + [c for c in [SKIP_COL, DEF_COL, "excel_row"]
                             if c in current_questions.columns]
    ref_cols = ["Q Name"] + ([DEF_COL] if has_def_ref else [])

    cur_rows = {r["Q Name"]: r
                for r in current_questions.select(cur_cols).iter_rows(named=True)}
    ref_rows = ({r["Q Name"]: r
                 for r in reference_questions.select(ref_cols).iter_rows(named=True)}
                if has_def_ref else {})

    # Q Name ordering for "next question" resolution
    qname_list = current_questions["Q Name"].to_list()
    qname_next = {q: qname_list[i + 1]
                  for i, q in enumerate(qname_list) if i + 1 < len(qname_list)}

    issues    = []
    seen_drift = set()

    for qname, cr in cur_rows.items():
        excel_row = cr.get("excel_row")
        skip_text = _normalize_skip_text(cr.get(SKIP_COL) or "")
        def_text  = _normalize_skip_text(cr.get(DEF_COL)  or "")

        # ── Layer 1: Skip Pattern vs Default column (consistency) ─────────────
        if has_def_cur and def_text and has_skip_cur:
            rules = _parse_default_skip_rules(def_text, curr_qnames)
            for desc in _check_skip_consistency(
                skip_text, rules, q_mandatory_map, curr_qnames, qname_next.get(qname)
            ):
                is_range = desc.startswith("option range mismatch")
                issues.append({
                    "issue_type": "skip_range_mismatch" if is_range else "skip_consistency_error",
                    "set_name": "",
                    "Q Name": qname, "field": SKIP_COL,
                    "current": skip_text[:220], "reference": def_text[:220],
                    "severity": "medium" if is_range else "high",
                    "excel_row": excel_row,
                })

        # ── Layer 2: Default column changed ref → current ─────────────────────
        if has_def_ref and def_text:
            ref_def = _normalize_skip_text(ref_rows.get(qname, {}).get(DEF_COL) or "")
            if ref_def and ref_def != def_text:
                issues.append({
                    "issue_type": "default_skip_modified", "set_name": "",
                    "Q Name": qname, "field": DEF_COL,
                    "current": def_text[:220], "reference": ref_def[:220],
                    "severity": "medium", "excel_row": excel_row,
                })

        # ── Layer 3: Option codes in Skip Pattern must exist in question options
        if has_skip_cur and skip_text:
            mentioned = _parse_skip_option_codes(skip_text)
            available = curr_opt_map.get(qname, set())
            if mentioned and available:
                invalid = sorted(c for c in mentioned if c not in available)
                if invalid:
                    issues.append({
                        "issue_type": "skip_pattern_invalid_option_code", "set_name": "",
                        "Q Name": qname, "field": SKIP_COL,
                        "current": f"invalid option code(s): {invalid}",
                        "reference": f"valid codes: {sorted(available)}",
                        "severity": "high", "excel_row": excel_row,
                    })

        # ── Layer 4: Option drift risk ─────────────────────────────────────────
        if has_skip_cur and skip_text and qname in option_issue_qnames:
            key = (qname, SKIP_COL)
            if key not in seen_drift:
                seen_drift.add(key)
                issues.append({
                    "issue_type": "potential_skip_pattern_option_drift", "set_name": "",
                    "Q Name": qname, "field": SKIP_COL,
                    "current": "Option set changed on a skip-driven question",
                    "reference": skip_text[:220],
                    "severity": "medium", "excel_row": excel_row,
                })

    if not issues:
        return pl.DataFrame(schema=EMPTY_SCHEMA)

    return (
        pl.DataFrame(issues)
        .unique(subset=["issue_type", "Q Name", "field", "current", "reference"], keep="first")
    )


## Step 8  Issue unifiers
These functions convert each comparison result into a common schema so they can
be stacked into a single `all_issues` table.

Common columns: `issue_type | set_name | Q Name | field | current | reference | severity | excel_row`

In [10]:
def make_presence_issues(added: pl.DataFrame, removed: pl.DataFrame) -> pl.DataFrame:
    """
    Converts added/removed question lists into the common issues schema.

    Severity rules:
      - Optional questions (o_ prefix): 'info'   tracked but not a blocker
      - Non-optional added            : 'medium'  unexpected addition, review needed
      - Non-optional removed          : 'high'    mandatory question missing
    """
    added_issues = added.with_columns([
        pl.lit("added_question").alias("issue_type"),
        pl.lit("").alias("set_name"),
        pl.lit("Q Name").alias("field"),
        pl.lit("present").alias("current"),
        pl.lit("missing_in_reference").alias("reference"),
        pl.lit("info").alias("severity"),
        pl.lit(None).cast(pl.Int64).alias("excel_row"),
    ]).select(["issue_type", "set_name", "Q Name", "field", "current", "reference", "severity", "excel_row"])

    removed_issues = removed.with_columns([
        pl.lit("removed_question").alias("issue_type"),
        pl.lit("").alias("set_name"),
        pl.lit("Q Name").alias("field"),
        pl.lit("missing_in_current").alias("current"),
        pl.lit("present").alias("reference"),
        pl.when(pl.col("is_optional")).then(pl.lit("info")).otherwise(pl.lit("high")).alias("severity"),
        pl.lit(None).cast(pl.Int64).alias("excel_row"),
    ]).select(["issue_type", "set_name", "Q Name", "field", "current", "reference", "severity", "excel_row"])

    return pl.concat([added_issues, removed_issues], how="vertical")


def make_mandatory_issues(mandatory_diff: pl.DataFrame) -> pl.DataFrame:
    """Converts mandatory mismatches into the common issues schema."""
    return (
        mandatory_diff
        .with_columns([
            pl.lit("").alias("set_name"),
            pl.col("Mandatory").alias("current"),
            pl.col("Mandatory_ref").alias("reference"),
            pl.lit("high").alias("severity"),
        ])
        .select(["issue_type", "set_name", "Q Name", "field", "current", "reference", "severity", "excel_row"])
    )


def make_option_issues(option_diff: pl.DataFrame) -> pl.DataFrame:
    """Converts option label mismatches into the common issues schema."""
    return (
        option_diff
        .with_columns([
            pl.lit("").alias("set_name"),
            pl.col("option_label").alias("current"),
            pl.col("option_label_ref").alias("reference"),
            pl.lit("medium").alias("severity"),
        ])
        .select(["issue_type", "set_name", "Q Name", "field", "current", "reference", "severity", "excel_row"])
    )


# Note: validate_critical_sets, validate_prefix_counts, validate_crop_harvest,
# and validate_skip_patterns already return the common schema directly.


def make_option_presence_issues(added_options: pl.DataFrame, removed_options: pl.DataFrame) -> pl.DataFrame:
    """Converts added/removed options into the common issues schema."""
    added_issues = added_options.with_columns([
        pl.lit("").alias("set_name"),
        pl.col("option_label").alias("current"),
        pl.lit("(not in template)").alias("reference"),
        pl.lit("medium").alias("severity"),
        pl.lit(None).cast(pl.Int64).alias("excel_row"),
    ]).select(["issue_type", "set_name", "Q Name", "field", "current", "reference", "severity", "excel_row"])

    removed_issues = removed_options.with_columns([
        pl.lit("").alias("set_name"),
        pl.lit("(removed)").alias("current"),
        pl.col("option_label").alias("reference"),
        pl.lit("high").alias("severity"),
        pl.lit(None).cast(pl.Int64).alias("excel_row"),
    ]).select(["issue_type", "set_name", "Q Name", "field", "current", "reference", "severity", "excel_row"])

    return pl.concat([added_issues, removed_issues], how="vertical")


def build_option_changes_view(
    en_option_diff: pl.DataFrame,
    en_added_options: pl.DataFrame,
    en_removed_options: pl.DataFrame,
    tgt_option_diff: pl.DataFrame,
    tgt_added_options: pl.DataFrame,
    tgt_removed_options: pl.DataFrame,
    target_lang: str,
) -> pl.DataFrame:
    """
    Builds Option Changes rows with independent EN and target-language checks.
    Q Name/field stay shared; EN and target value pairs are kept side by side.
    """
    is_en = str(target_lang).upper() == "EN"

    def _scope_view(option_diff, added_opts, removed_opts, scope):
        mismatch = option_diff.with_columns([
            pl.col("option_label").alias("current"),
            pl.col("option_label_ref").alias("reference"),
            pl.col("excel_row").cast(pl.Int64).alias("excel_row"),
        ]).select(["issue_type", "Q Name", "field", "current", "reference", "severity", "excel_row"])

        added = added_opts.with_columns([
            pl.col("option_label").alias("current"),
            pl.lit("(not in template)").alias("reference"),
            pl.lit(None).cast(pl.Int64).alias("excel_row"),
        ]).select(["issue_type", "Q Name", "field", "current", "reference", "severity", "excel_row"])

        removed = removed_opts.with_columns([
            pl.lit("(removed)").alias("current"),
            pl.col("option_label").alias("reference"),
            pl.lit(None).cast(pl.Int64).alias("excel_row"),
        ]).select(["issue_type", "Q Name", "field", "current", "reference", "severity", "excel_row"])

        base = pl.concat([mismatch, added, removed], how="vertical")
        return base.rename({
            "current": f"current_{scope.lower()}",
            "reference": f"reference_{scope.lower()}",
            "severity": f"severity_{scope.lower()}",
            "excel_row": f"excel_row_{scope.lower()}",
        })

    en_view = _scope_view(en_option_diff, en_added_options, en_removed_options, "EN")
    if is_en:
        return (
            en_view
            .with_columns([
                pl.lit("").alias("set_name"),
                pl.col("current_en").alias("current"),
                pl.col("reference_en").alias("reference"),
                pl.col("severity_en").alias("severity"),
                pl.col("excel_row_en").alias("excel_row"),
            ])
            .select(["issue_type", "set_name", "Q Name", "field", "current", "reference", "severity", "excel_row"])
        )

    tgt_view = _scope_view(tgt_option_diff, tgt_added_options, tgt_removed_options, "TGT")
    keys = ["issue_type", "Q Name", "field"]
    out = en_view.join(tgt_view, on=keys, how="full", suffix="_t")

    return (
        out
        .with_columns([
            pl.coalesce([pl.col("issue_type"), pl.col("issue_type_t")]).alias("issue_type"),
            pl.coalesce([pl.col("Q Name"), pl.col("Q Name_t")]).alias("Q Name"),
            pl.coalesce([pl.col("field"), pl.col("field_t")]).alias("field"),
            pl.lit("").alias("set_name"),
            pl.coalesce([pl.col("current_tgt"), pl.lit("")]).alias("current"),
            pl.coalesce([pl.col("reference_tgt"), pl.lit("")]).alias("reference"),
            pl.coalesce([pl.col("current_en"), pl.lit("")]).alias("current_en"),
            pl.coalesce([pl.col("reference_en"), pl.lit("")]).alias("reference_en"),
            pl.when(pl.col("severity_tgt") == "high").then(pl.lit("high"))
            .when(pl.col("severity_en") == "high").then(pl.lit("high"))
            .when(pl.col("severity_tgt") == "medium").then(pl.lit("medium"))
            .when(pl.col("severity_en") == "medium").then(pl.lit("medium"))
            .otherwise(pl.lit("info")).alias("severity"),
            pl.coalesce([pl.col("excel_row_tgt"), pl.col("excel_row_en")]).alias("excel_row"),
        ])
        .select([
            "issue_type", "set_name", "Q Name", "field", "current", "reference", "current_en", "reference_en",
            "severity", "excel_row",
        ])
    )




## Step 9  Run the full pipeline
Runs all seven checks and assembles a single `all_issues` table sorted by severity (**high  medium  info**).

| Check | Severity |
|---|---|
| Removed non-optional question | high |
| Mandatory flag changed | high |
| Critical set question missing or wrong | high |
| CS count below minimum (4 stress / 3 crisis / 3 emergency) | high |
| Crop harvest rule violated | high |
| Skip pattern references missing question | high |
| Added non-optional question | medium |
| Option label changed | medium |
| Optional (`o_*`) question added or removed | info |

To update which question groups are critical or what the minimum counts are, edit **`critical_sets.yaml`**  no code changes needed.


In [11]:
def compare_question_presence(
    current_questions : pl.DataFrame,
    reference_questions: pl.DataFrame,
) -> tuple[pl.DataFrame, pl.DataFrame]:
    """
    Returns (added, removed). Each DataFrame includes `is_optional`.
    """
    key_col = "Q Name"

    added = (
        current_questions.select([key_col])
        .join(reference_questions.select([key_col]), on=key_col, how="anti")
        .with_columns(pl.col(key_col).str.starts_with("o_").alias("is_optional"))
    )

    removed = (
        reference_questions.select([key_col])
        .join(current_questions.select([key_col]), on=key_col, how="anti")
        .with_columns(pl.col(key_col).str.starts_with("o_").alias("is_optional"))
    )

    return added, removed


def compare_mandatory(
    current_questions : pl.DataFrame,
    reference_questions: pl.DataFrame,
    treat_blank_as_no: bool = True,
) -> pl.DataFrame:
    """
    Compares Mandatory column values by Q Name.
    """
    if "Mandatory" not in current_questions.columns or "Mandatory" not in reference_questions.columns:
        return pl.DataFrame(schema={
            "issue_type": pl.Utf8, "Q Name": pl.Utf8, "field": pl.Utf8,
            "Mandatory": pl.Utf8, "Mandatory_ref": pl.Utf8, "excel_row": pl.Int64,
        })

    joined = (
        current_questions.select(["Q Name", "Mandatory", "excel_row"])
        .join(reference_questions.select(["Q Name", "Mandatory"]), on="Q Name", how="inner", suffix="_ref")
        .with_columns([
            pl.col("Mandatory").cast(pl.Utf8).fill_null("").str.strip_chars().alias("Mandatory"),
            pl.col("Mandatory_ref").cast(pl.Utf8).fill_null("").str.strip_chars().alias("Mandatory_ref"),
        ])
    )

    if treat_blank_as_no:
        joined = joined.with_columns([
            pl.when(pl.col("Mandatory") == "").then(pl.lit("No")).otherwise(pl.col("Mandatory")).alias("Mandatory"),
            pl.when(pl.col("Mandatory_ref") == "").then(pl.lit("No")).otherwise(pl.col("Mandatory_ref")).alias("Mandatory_ref"),
        ])

    return (
        joined
        .filter(normalize_text_expr("Mandatory") != normalize_text_expr("Mandatory_ref"))
        .with_columns([
            pl.lit("mandatory_column_mismatch").alias("issue_type"),
            pl.lit("Mandatory").alias("field"),
        ])
        .select(["issue_type", "Q Name", "field", "Mandatory", "Mandatory_ref", "excel_row"])
    )


def compare_option_labels_single(
    current_options : pl.DataFrame,
    reference_options: pl.DataFrame,
    lang_scope: str,
) -> pl.DataFrame:
    """
    Option label changes for matching Q Name + option_code.
    """
    return (
        current_options
        .join(reference_options, on=["Q Name", "option_code"], how="inner", suffix="_ref")
        .with_columns([
            normalize_text_expr("option_label").alias("option_label_norm"),
            normalize_text_expr("option_label_ref").alias("option_label_ref_norm"),
        ])
        .filter(pl.col("option_label_norm") != pl.col("option_label_ref_norm"))
        .with_columns([
            pl.lit("option_label_mismatch").alias("issue_type"),
            pl.concat_str([pl.lit("option_"), pl.col("option_code").cast(pl.Utf8)]).alias("field"),
            pl.lit("medium").alias("severity"),
            pl.lit(lang_scope).alias("lang_scope"),
        ])
        .select(["issue_type", "Q Name", "field", "option_label", "option_label_ref", "severity", "excel_row", "lang_scope"])
    )


def compare_option_presence_single(
    current_options : pl.DataFrame,
    reference_options: pl.DataFrame,
    lang_scope: str,
) -> tuple[pl.DataFrame, pl.DataFrame]:
    """
    Added/removed options for one language scope.
    """
    key_cols = ["Q Name", "option_code"]

    removed_options = (
        reference_options.select(key_cols + ["option_label"])
        .join(current_options.select(key_cols), on=key_cols, how="anti")
        .with_columns([
            pl.lit("removed_option").alias("issue_type"),
            pl.concat_str([pl.lit("option_"), pl.col("option_code").cast(pl.Utf8)]).alias("field"),
            pl.lit("high").alias("severity"),
            pl.lit(lang_scope).alias("lang_scope"),
        ])
        .select(["issue_type", "Q Name", "field", "option_label", "severity", "lang_scope"])
    )

    added_options = (
        current_options.select(key_cols + ["option_label"])
        .join(reference_options.select(key_cols), on=key_cols, how="anti")
        .with_columns([
            pl.lit("added_option").alias("issue_type"),
            pl.concat_str([pl.lit("option_"), pl.col("option_code").cast(pl.Utf8)]).alias("field"),
            pl.lit("medium").alias("severity"),
            pl.lit(lang_scope).alias("lang_scope"),
        ])
        .select(["issue_type", "Q Name", "field", "option_label", "severity", "lang_scope"])
    )

    return added_options, removed_options




In [12]:
def validate_critical_sets(
    questions_df: pl.DataFrame,
    exact_sets  : dict,
) -> pl.DataFrame:
    """
    Checks that each critical question group is fully present with the expected
    Mandatory value. Rules come from critical_sets.yaml -> exact_sets.

    required=true  -> missing = issue_type 'missing_critical_question', severity HIGH
    required=false -> missing = issue_type 'advisory_question',          severity MEDIUM
    """
    EMPTY_SCHEMA = {
        "issue_type": pl.Utf8, "set_name": pl.Utf8, "Q Name": pl.Utf8,
        "field": pl.Utf8, "current": pl.Utf8, "reference": pl.Utf8,
        "severity": pl.Utf8, "excel_row": pl.Int64,
    }

    if not exact_sets:
        return pl.DataFrame(schema=EMPTY_SCHEMA)

    if "Mandatory_norm" not in questions_df.columns:
        questions_df = questions_df.with_columns(
            normalize_mandatory_expr("Mandatory").alias("Mandatory_norm")
        )

    present_qnames = set(questions_df["Q Name"].to_list())
    issues = []

    for set_name, rules in exact_sets.items():
        required_names   = [r["q_name"] for r in rules if r.get("required", True)]
        present_in_set   = [r["q_name"] for r in rules if r["q_name"] in present_qnames]
        present_required = [q for q in required_names if q in present_qnames]

        if 0 < len(present_required) < len(required_names):
            issues.append({
                "issue_type": "partial_critical_set",
                "set_name"  : set_name, "Q Name": "",
                "field"     : "Q Name",
                "current"   : ", ".join(sorted(present_in_set)),
                "reference" : f"Expected all {len(required_names)} required questions",
                "severity"  : "high", "excel_row": None,
            })

        for rule in rules:
            q_name   = rule["q_name"]
            expected = rule.get("expected_mandatory", "")
            required = rule.get("required", True)

            if q_name not in present_qnames:
                issues.append({
                    "issue_type": "missing_critical_question" if required else "advisory_question",
                    "set_name"  : set_name, "Q Name": q_name,
                    "field"     : "Q Name", "current": "",
                    "reference" : "present",
                    "severity"  : "high" if required else "medium",
                    "excel_row" : None,
                })
                continue

            if not expected:
                continue

            row = (
                questions_df
                .filter(pl.col("Q Name") == q_name)
                .select(["Q Name", "Mandatory", "Mandatory_norm", "excel_row"])
                .to_dicts()
            )
            if not row:
                continue
            row = row[0]
            if row["Mandatory_norm"] != expected:
                issues.append({
                    "issue_type": "critical_mandatory_mismatch",
                    "set_name"  : set_name, "Q Name": q_name,
                    "field"     : "Mandatory",
                    "current"   : row["Mandatory"], "reference": expected,
                    "severity"  : "high", "excel_row": row.get("excel_row"),
                })

    if not issues:
        return pl.DataFrame(schema=EMPTY_SCHEMA)
    return pl.DataFrame(issues)


def validate_prefix_counts(
    current_questions: pl.DataFrame,
    min_count_sets   : dict,
) -> pl.DataFrame:
    """
    Checks that questions with a given prefix appear at least min_count times.
    Covers CS groups (cs_stress_*, cs_crisis_*, cs_emergency_*) and HDDS count.
    Rules come from critical_sets.yaml -> min_count_sets.
    """
    EMPTY_SCHEMA = {
        "issue_type": pl.Utf8, "set_name": pl.Utf8, "Q Name": pl.Utf8,
        "field": pl.Utf8, "current": pl.Utf8, "reference": pl.Utf8,
        "severity": pl.Utf8, "excel_row": pl.Int64,
    }
    if not min_count_sets:
        return pl.DataFrame(schema=EMPTY_SCHEMA)

    all_qnames = current_questions["Q Name"].to_list()
    issues = []
    for set_name, rule in min_count_sets.items():
        prefix    = rule.get("prefix", "")
        min_count = rule.get("min_count", 1)
        desc      = rule.get("description", f"At least {min_count} '{prefix}*' questions required")
        matched   = sorted(q for q in all_qnames if q.startswith(prefix))
        if len(matched) < min_count:
            found_str = f"{len(matched)} found" + (f": {', '.join(matched)}" if matched else " (none)")
            issues.append({
                "issue_type": "below_minimum_count", "set_name": set_name, "Q Name": "",
                "field": "count", "current": found_str, "reference": desc,
                "severity": "high", "excel_row": None,
            })

    if not issues:
        return pl.DataFrame(schema=EMPTY_SCHEMA)
    return pl.DataFrame(issues)


def validate_crop_harvest(
    current_questions: pl.DataFrame,
    crop_rules       : dict,
) -> pl.DataFrame:
    """
    Questionnaire must contain EITHER only the minimal set OR all questions in
    the full set. Rules come from critical_sets.yaml -> crop_harvest.
    """
    EMPTY_SCHEMA = {
        "issue_type": pl.Utf8, "set_name": pl.Utf8, "Q Name": pl.Utf8,
        "field": pl.Utf8, "current": pl.Utf8, "reference": pl.Utf8,
        "severity": pl.Utf8, "excel_row": pl.Int64,
    }
    if not crop_rules:
        return pl.DataFrame(schema=EMPTY_SCHEMA)

    minimal = set(crop_rules.get("minimal", []))
    full    = set(crop_rules.get("full", []))
    if not minimal and not full:
        return pl.DataFrame(schema=EMPTY_SCHEMA)

    all_qnames     = set(current_questions["Q Name"].to_list())
    relevant_found = {q for q in all_qnames if q in full or q in minimal}

    if relevant_found == minimal or full.issubset(all_qnames):
        return pl.DataFrame(schema=EMPTY_SCHEMA)

    issues = [{
        "issue_type": "crop_harvest_violation", "set_name": "CRP_HARV", "Q Name": "",
        "field": "Q Name",
        "current"  : f"found: {', '.join(sorted(relevant_found)) or 'none'}",
        "reference": f"either only [{', '.join(sorted(minimal))}] OR all of [{', '.join(sorted(full))}]",
        "severity" : "high", "excel_row": None,
    }]
    return pl.DataFrame(issues)




In [13]:
#  Run all comparisons
import time
_t0 = time.perf_counter()
print("[timing] run block started", flush=True)
_t_prev = _t0
def _tick(label: str):
    global _t_prev
    _now = time.perf_counter()
    print(f"[timing] {label:<34} {_now - _t_prev:8.3f}s", flush=True)
    _t_prev = _now

added, removed = compare_question_presence(current_questions, reference_questions)
_tick("compare_question_presence")

mandatory_diff = compare_mandatory(
    current_questions,
    reference_questions,
    treat_blank_as_no=cfg.treat_blank_as_no,
)
_tick("compare_mandatory")

en_option_diff = compare_option_labels_single(current_options_en, reference_options_en, "EN")
en_added_opts, en_removed_opts = compare_option_presence_single(current_options_en, reference_options_en, "EN")
_tick("compare options EN")

if target_lang == "EN":
    tgt_option_diff = en_option_diff.clone()
    tgt_added_opts = en_added_opts.clone()
    tgt_removed_opts = en_removed_opts.clone()
else:
    tgt_option_diff = compare_option_labels_single(current_options_tgt, reference_options_tgt, target_lang)
    tgt_added_opts, tgt_removed_opts = compare_option_presence_single(current_options_tgt, reference_options_tgt, target_lang)
_tick("compare options target")

critical_issues = validate_critical_sets(current_questions, rules["exact_sets"])
count_issues = validate_prefix_counts(current_questions, rules["min_count_sets"])
harvest_issues = validate_crop_harvest(current_questions, rules["crop_harvest"])
_tick("critical/count/harvest checks")

skip_option_issue_qnames = (
    set(en_option_diff["Q Name"].to_list())
    | set(en_added_opts["Q Name"].to_list())
    | set(en_removed_opts["Q Name"].to_list())
)
if target_lang != "EN":
    skip_option_issue_qnames |= (
        set(tgt_option_diff["Q Name"].to_list())
        | set(tgt_added_opts["Q Name"].to_list())
        | set(tgt_removed_opts["Q Name"].to_list())
    )

# Keep option-drift risk focused on shared questions only.
# Added/removed questions are already reported in structure checks.
shared_qnames_for_drift = (
    set(current_questions["Q Name"].to_list())
    & set(reference_questions["Q Name"].to_list())
)
skip_option_issue_qnames &= shared_qnames_for_drift

# Build mandatory map needed by validate_skip_patterns (layer 1 flexibility check)
_q_mand_map: dict[str, str] = {}
for _df in [reference_questions, current_questions]:
    for _r in _df.select(["Q Name", "Mandatory"]).iter_rows(named=True):
        _q = _r["Q Name"]
        _m = str(_r.get("Mandatory") or "").strip().lower()
        if str(_q).startswith("o_"):
            _q_mand_map[_q] = "optional"
        elif "panel" in _m:
            _q_mand_map[_q] = "mandatory-panel"
        elif _m in ("yes", "y", "true", "1"):
            _q_mand_map[_q] = "mandatory"
        else:
            _q_mand_map[_q] = "non-mandatory"
_tick("mandatory map")

skip_issues = validate_skip_patterns(
    current_questions,
    reference_questions,
    current_options_en=current_options_en,
    reference_options_en=reference_options_en,
    option_issue_qnames=skip_option_issue_qnames,
    q_mandatory_map=_q_mand_map,
)
_tick("validate_skip_patterns")


#  Convert to common schema
presence_issues = make_presence_issues(added, removed)
mandatory_issues = make_mandatory_issues(mandatory_diff)
option_changes_view = build_option_changes_view(
    en_option_diff, en_added_opts, en_removed_opts,
    tgt_option_diff, tgt_added_opts, tgt_removed_opts,
    target_lang,
)
option_issues = option_changes_view.select([
    "issue_type", "set_name", "Q Name", "field", "current", "reference", "severity", "excel_row"
])
option_presence_issues = option_issues.filter(pl.col("issue_type").is_in(["added_option", "removed_option"]))
_tick("build option/common issue frames")

#  Filter option issues: only report options for questions in BOTH files
_excluded_qnames = set(added["Q Name"].to_list()) | set(removed["Q Name"].to_list())
if _excluded_qnames:
    _excl = list(_excluded_qnames)
    option_issues = option_issues.filter(~pl.col("Q Name").is_in(_excl))
    option_presence_issues = option_presence_issues.filter(~pl.col("Q Name").is_in(_excl))
    option_changes_view = option_changes_view.filter(~pl.col("Q Name").is_in(_excl))
_tick("filter option issues")

#  Stack all issues
all_issues = pl.concat(
    [presence_issues, mandatory_issues, option_issues, option_presence_issues,
     critical_issues, count_issues, harvest_issues, skip_issues],
    how="vertical",
)

#  Sort: high -> medium -> info
all_issues = (
    all_issues
    .with_columns(
        pl.when(pl.col("severity") == "high").then(pl.lit(0))
        .when(pl.col("severity") == "medium").then(pl.lit(1))
        .otherwise(pl.lit(2))
        .alias("_sort_order")
    )
    .sort(["_sort_order", "issue_type", "Q Name"])
    .drop("_sort_order")
)
_tick("concat + severity sort")

#  CS downgrade: if count minimum is met, removed CS questions -> info
_passing_prefixes = []
for _set_name, _rule in rules.get("min_count_sets", {}).items():
    if count_issues.filter(pl.col("set_name") == _set_name).height == 0:
        _prefix = _rule.get("prefix", "")
        if _prefix:
            _passing_prefixes.append(_prefix)

if _passing_prefixes:
    _starts_exprs = [pl.col("Q Name").cast(pl.Utf8).str.starts_with(p) for p in _passing_prefixes]
    _q_matches_prefix = pl.any_horizontal(_starts_exprs) if len(_starts_exprs) > 1 else _starts_exprs[0]
    all_issues = all_issues.with_columns(
        pl.when(
            (pl.col("issue_type") == "removed_question") & _q_matches_prefix
        ).then(pl.lit("info"))
        .otherwise(pl.col("severity"))
        .alias("severity")
    )
_tick("cs downgrade")

#  Add mandatory_cat via Polars lookup (current values override reference values)
_ref_m = reference_questions.select(["Q Name", "Mandatory"]).with_columns(pl.lit(0).alias("_prio"))
_cur_m = current_questions.select(["Q Name", "Mandatory"]).with_columns(pl.lit(1).alias("_prio"))
_mand_lookup = (
    pl.concat([_ref_m, _cur_m], how="vertical")
    .with_columns([
        pl.col("Q Name").cast(pl.Utf8).alias("Q Name"),
        pl.col("Mandatory").cast(pl.Utf8).fill_null("").str.strip_chars().str.to_lowercase().alias("_m"),
    ])
    .with_columns([
        pl.when(pl.col("Q Name").str.starts_with("o_")).then(pl.lit("optional"))
        .when(pl.col("_m").str.contains("panel")).then(pl.lit("mandatory-panel"))
        .when(pl.col("_m").is_in(["yes", "y", "true", "1"])).then(pl.lit("mandatory"))
        .otherwise(pl.lit("non-mandatory"))
        .alias("mandatory_cat")
    ])
    .sort(["Q Name", "_prio"])
    .group_by("Q Name")
    .agg(pl.col("mandatory_cat").last().alias("mandatory_cat"))
)

all_issues = (
    all_issues
    .join(_mand_lookup, on="Q Name", how="left")
    .with_columns(pl.col("mandatory_cat").fill_null(""))
)

option_changes_view = (
    option_changes_view
    .join(_mand_lookup, on="Q Name", how="left")
    .with_columns(pl.col("mandatory_cat").fill_null(""))
)
_tick("mandatory_cat lookup join")

option_changes_view = (
    option_changes_view
    .with_columns(
        pl.when(pl.col("severity") == "high").then(pl.lit(0))
        .when(pl.col("severity") == "medium").then(pl.lit(1))
        .otherwise(pl.lit(2))
        .alias("_s")
    )
    .sort(["_s", "Q Name", "field"])
    .drop("_s")
)
_tick("sort option_changes_view")

#  Build found_info for the Critical Sets sheet
_found_info = {}
for _set_name, _rule in rules.get("min_count_sets", {}).items():
    _prefix = _rule.get("prefix", "")
    _matched = sorted(q for q in current_questions["Q Name"].to_list() if q.startswith(_prefix))
    _found_info[_set_name] = _matched
_tick("build found_info")

#  Summary
print(f"Questions added       : {added.height}")
print(f"Questions removed     : {removed.height}")
print(f"Mandatory mismatches  : {mandatory_diff.height}")
print(f"Option label changes  : {option_issues.filter(pl.col('issue_type') == 'option_label_mismatch').height}")
print(f"Options removed       : {option_issues.filter(pl.col('issue_type') == 'removed_option').height}")
print(f"Options added         : {option_issues.filter(pl.col('issue_type') == 'added_option').height}")
print(f"Critical set issues   : {critical_issues.height}")
print(f"Count rule violations : {count_issues.height}")
print(f"Crop harvest issues   : {harvest_issues.height}")
print(f"Skip pattern issues   : {skip_issues.height}")
print(f"")
print(f"Total issues          : {all_issues.height}")
print(f"[timing] TOTAL run block                    {time.perf_counter() - _t0:8.3f}s", flush=True)

if all_issues.height > 0:
    print("\nFirst 20 issues:")
    print(all_issues.head(20))
else:
    print("\nNo issues found.")




[timing] run block started
[timing] compare_question_presence             0.010s
[timing] compare_mandatory                     0.017s
[timing] compare options EN                    0.028s
[timing] compare options target                0.011s
[timing] critical/count/harvest checks         0.003s
[timing] mandatory map                         0.003s
[timing] validate_skip_patterns                0.012s
[timing] build option/common issue frames      0.013s
[timing] filter option issues                  0.002s
[timing] concat + severity sort                0.006s
[timing] cs downgrade                          0.005s
[timing] mandatory_cat lookup join             0.012s
[timing] sort option_changes_view              0.002s
[timing] build found_info                      0.001s
Questions added       : 30
Questions removed     : 14
Mandatory mismatches  : 1
Option label changes  : 35
Options removed       : 3
Options added         : 4
Critical set issues   : 1
Count rule violations : 0
Crop h

## Step 10  Export to Excel

Produces a 4-sheet report:

| Sheet | What it shows |
|---|---|
| **Summary** | Per-set  PASS /  FAIL status at a glance, plus issue counts by category |
| **Critical Sets** | Full detail for all structural issues: CS coping strategies, HDDS, WEALTH, RCSI, crop harvest, broken skip patterns |
| **Template Changes** | Mandatory flag changes, added/removed non-optional questions, option label changes |
| **Optional Questions** | `o_*` questions added or removed (informational only) |

Colour coding:  red = high severity,  orange = medium,  blue = info,  green = PASS.


In [14]:
from shutil import copyfile
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

#  Colours 
FILL_HIGH    = PatternFill("solid", fgColor="F4CCCC")
FILL_MEDIUM  = PatternFill("solid", fgColor="FCE5CD")
FILL_INFO    = PatternFill("solid", fgColor="CFE2F3")
FILL_PASS    = PatternFill("solid", fgColor="D9EAD3")
FILL_HEADER  = PatternFill("solid", fgColor="274E13")
FILL_SECTION = PatternFill("solid", fgColor="E8F5E9")

FONT_TITLE   = Font(bold=True, size=13, color="274E13")
FONT_HEADER  = Font(bold=True, color="FFFFFF", size=11)
FONT_SECTION = Font(bold=True, color="274E13", size=11)
FONT_NORMAL  = Font(size=10)

THIN_BORDER = Border(
    left=Side(style="thin", color="CCCCCC"),
    right=Side(style="thin", color="CCCCCC"),
    bottom=Side(style="thin", color="CCCCCC"),
)

SEVERITY_FILL = {"high": FILL_HIGH, "medium": FILL_MEDIUM, "info": FILL_INFO}
STATUS_FILL   = {"PASS": FILL_PASS, "FAIL": FILL_HIGH, "WARNING": FILL_MEDIUM}

CRITICAL_SET_ISSUE_TYPES = {
    "missing_critical_question", "advisory_question", "partial_critical_set",
    "critical_mandatory_mismatch", "below_minimum_count", "crop_harvest_violation",
}

SKIP_PATTERN_ISSUE_TYPES = {
    "skip_consistency_error",
    "skip_range_mismatch",
    "default_skip_modified",
    "skip_pattern_invalid_option_code",
    "potential_skip_pattern_option_drift",
}

#  Shared helpers 
def _header_row(ws, row, values):
    for col, val in enumerate(values, 1):
        c = ws.cell(row=row, column=col, value=val)
        c.fill = FILL_HEADER; c.font = FONT_HEADER
        c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[row].height = 18

def _section_header(ws, row, title, n_cols):
    ws.merge_cells(start_row=row, start_column=1, end_row=row, end_column=n_cols)
    c = ws.cell(row=row, column=1, value=title)
    c.fill = FILL_SECTION; c.font = FONT_SECTION
    c.alignment = Alignment(horizontal="left", vertical="center")
    ws.row_dimensions[row].height = 20

def _data_row(ws, row, n_cols, severity="", fill=None):
    use_fill = fill or SEVERITY_FILL.get(severity)
    for col in range(1, n_cols + 1):
        c = ws.cell(row=row, column=col)
        c.font = FONT_NORMAL; c.border = THIN_BORDER
        c.alignment = Alignment(vertical="top", wrap_text=True)
        if use_fill:
            c.fill = use_fill

def _autofit(ws, mn=12, mx=55):
    for col_cells in ws.columns:
        w = max((len(str(c.value)) if c.value else 0) for c in col_cells)
        ws.column_dimensions[get_column_letter(col_cells[0].column)].width = min(max(w + 2, mn), mx)

def _issues_table(ws, start_row, df):
    """Renders a header + data table. Only columns present in df are shown."""
    COL_MAP = [
        ("issue_type",    "Issue type"),
        ("mandatory_cat", "Type"),
        ("set_name",      "Set"),
        ("Q Name",        "Q Name"),
        ("field",         "Field"),
        ("current",       "Current value"),
        ("reference",     "Reference / rule"),
        ("current_en",    "Current value (EN)"),
        ("reference_en",  "Reference / rule (EN)"),
        ("severity",      "Severity"),
        ("excel_row",     "Excel row"),
    ]
    cols = [(s, d) for s, d in COL_MAP if s in df.columns]
    _header_row(ws, start_row, [d for _, d in cols])
    r = start_row + 1
    if df.height == 0:
        c = ws.cell(row=r, column=1, value="No issues in this category")
        c.font = Font(bold=True, color="274E13", size=10)
        return r + 2
    for row_data in df.to_dicts():
        for col, (src, _) in enumerate(cols, 1):
            ws.cell(row=r, column=col, value=row_data.get(src))
        _data_row(ws, r, len(cols), row_data.get("severity", ""))
        r += 1
    ws.freeze_panes = f"A{start_row + 1}"
    ws.auto_filter.ref = f"A{start_row}:{get_column_letter(len(cols))}{start_row}"
    return r + 1


#  Per-set status builder 
def _set_status(all_issues, rules):
    results = []
    for set_name in rules.get("exact_sets", {}).keys():
        si   = all_issues.filter(pl.col("set_name") == set_name)
        high = si.filter(pl.col("severity") == "high").height
        med  = si.filter(pl.col("severity") == "medium").height
        if high > 0:
            status  = "FAIL"
            details = "; ".join(f"{r['Q Name'] or r['current']}" for r in si.filter(pl.col("severity") == "high").to_dicts())
        elif med > 0:
            status  = "WARNING"
            details = "; ".join(f"{r['Q Name'] or r['current']}" for r in si.to_dicts())
        else:
            status, details = "PASS", "All required questions present and correct"
        results.append({"set": set_name, "status": status, "details": details})
    for set_name, rule in rules.get("min_count_sets", {}).items():
        si = all_issues.filter(pl.col("set_name") == set_name)
        if si.filter(pl.col("severity").is_in(["high", "medium"])).height > 0:
            status, details = "FAIL", si["current"].to_list()[0]
        else:
            status  = "PASS"
            details = f">={rule.get('min_count',0)} {rule.get('prefix','')}* questions confirmed"
        results.append({"set": set_name, "status": status, "details": details})
    if rules.get("crop_harvest"):
        crp = all_issues.filter(pl.col("set_name") == "CRP_HARV")
        if crp.height > 0:
            status, details = "FAIL", crp["current"].to_list()[0]
        else:
            status, details = "PASS", "Crop harvest rule respected"
        results.append({"set": "CRP_HARV", "status": status, "details": details})

    skip_fail_types = {
        "skip_consistency_error",
        "skip_pattern_invalid_option_code",
    }
    skip_warn_types = {
        "potential_skip_pattern_option_drift",
        "default_skip_modified",
        "skip_range_mismatch",
    }

    skip_fail = all_issues.filter(pl.col("issue_type").is_in(list(skip_fail_types)))
    skip_warn = all_issues.filter(pl.col("issue_type").is_in(list(skip_warn_types)))

    if skip_fail.height > 0:
        skip_status = "FAIL"
        skip_details = f"{skip_fail.height} critical skip-pattern issue(s); {skip_warn.height} potential risk(s)"
    elif skip_warn.height > 0:
        skip_status = "WARNING"
        skip_details = f"{skip_warn.height} potential skip-pattern risk(s) from option changes"
    else:
        skip_status = "PASS"
        skip_details = "All skip pattern checks passed"

    results.append({
        "set"    : "SKIP_PATTERNS",
        "status" : skip_status,
        "details": skip_details,
    })
    return results


def _cat_counts(df):
    """Returns counts by mandatory_cat for the four standard categories."""
    cats = ["mandatory", "mandatory-panel", "non-mandatory", "optional"]
    if "mandatory_cat" not in df.columns or df.height == 0:
        return {c: 0 for c in cats}
    return {cat: df.filter(pl.col("mandatory_cat") == cat).height for cat in cats}


#  Sheet 1: Summary 
def write_summary_sheet(wb, all_issues, rules):
    ws = wb.create_sheet("Summary")
    ws.sheet_view.showGridLines = False
    # A=category/set  B=total/status  C=mandatory/details  D=m-panel  E=non-mand.  F=optional  G=severity
    for col, w in zip("ABCDEFG", [32, 8, 13, 10, 13, 10, 14]):
        ws.column_dimensions[col].width = w

    ws.merge_cells("A1:G1")
    ws["A1"] = "Questionnaire Validation Report"
    ws["A1"].font = FONT_TITLE
    ws["A1"].alignment = Alignment(horizontal="left", vertical="center")
    ws.row_dimensions[1].height = 26

    r = 3
    #  Critical sets 
    _section_header(ws, r, "CRITICAL SETS STATUS", 7); r += 1
    _header_row(ws, r, ["Critical set", "Status", "Details", "", "", "", ""]); r += 1
    for row_data in _set_status(all_issues, rules):
        fill = STATUS_FILL.get(row_data["status"])
        ws.cell(row=r, column=1, value=row_data["set"])
        ws.cell(row=r, column=2, value=row_data["status"])
        ws.cell(row=r, column=3, value=row_data["details"])
        _data_row(ws, r, 7, fill=fill); r += 1

    r += 1
    #  Question changes (7 cols with mandatory breakdown) 
    _section_header(ws, r, "QUESTION CHANGES", 7); r += 1
    _header_row(ws, r, ["Category", "Total", "Mandatory", "M-Panel", "Non-mand.", "Optional", "Severity"]); r += 1
    for label, itype, sev_filter, disp_sev in [
        ("Mandatory flag changes",     "mandatory_column_mismatch", None,   "high"),
        ("Questions removed (high)",   "removed_question",   "high", "high"),
        ("Questions removed (info)",   "removed_question",   "info", "info"),
        ("Questions added",            "added_question",     None,   "info"),
    ]:
        q = all_issues.filter(pl.col("issue_type") == itype)
        if sev_filter:
            q = q.filter(pl.col("severity") == sev_filter)
        counts = _cat_counts(q)
        ws.cell(row=r, column=1, value=label)
        ws.cell(row=r, column=2, value=q.height)
        ws.cell(row=r, column=3, value=counts["mandatory"])
        ws.cell(row=r, column=4, value=counts["mandatory-panel"])
        ws.cell(row=r, column=5, value=counts["non-mandatory"])
        ws.cell(row=r, column=6, value=counts["optional"])
        ws.cell(row=r, column=7, value=disp_sev)
        _data_row(ws, r, 7, disp_sev); r += 1

    r += 1
    #  Option changes (7 cols with mandatory breakdown) 
    _section_header(ws, r, "OPTION CHANGES (questions present in both files)", 7); r += 1
    _header_row(ws, r, ["Category", "Total", "Mandatory", "M-Panel", "Non-mand.", "Optional", "Severity"]); r += 1
    for label, itype, disp_sev in [
        ("Options removed",       "removed_option",        "high"),
        ("Options added",         "added_option",          "medium"),
        ("Option labels changed", "option_label_mismatch", "medium"),
    ]:
        q = all_issues.filter(pl.col("issue_type") == itype)
        counts = _cat_counts(q)
        ws.cell(row=r, column=1, value=label)
        ws.cell(row=r, column=2, value=q.height)
        ws.cell(row=r, column=3, value=counts["mandatory"])
        ws.cell(row=r, column=4, value=counts["mandatory-panel"])
        ws.cell(row=r, column=5, value=counts["non-mandatory"])
        ws.cell(row=r, column=6, value=counts["optional"])
        ws.cell(row=r, column=7, value=disp_sev)
        _data_row(ws, r, 7, disp_sev); r += 1


#  Sheet 2: Structure/SkipPattern 
def write_structure_skip_sheet(wb, all_issues: pl.DataFrame):
    ws = wb.create_sheet("Structure-SkipPattern")
    ws.sheet_view.showGridLines = False

    df = all_issues.filter(
        pl.col("issue_type").is_in(list(SKIP_PATTERN_ISSUE_TYPES))
    )
    df = (
        df.with_columns(
            pl.when(pl.col("severity") == "high").then(pl.lit(0))
            .when(pl.col("severity") == "medium").then(pl.lit(1))
            .otherwise(pl.lit(2))
            .alias("_s")
        )
        .sort(["_s", "issue_type", "Q Name"])
        .drop("_s")
    )

    _section_header(ws, 1, "STRUCTURE-SKIP PATTERN  Skip pattern issues", 8)

    if df.height == 0:
        next_row = 2
        ws.cell(row=next_row, column=1, value="No skip pattern issues found")
        _data_row(ws, next_row, 8, "info")
        _autofit(ws)
        return

    _issues_table(ws, 2, df)
    ws.freeze_panes = "A3"
    _autofit(ws)


#  Sheet 2: Critical Sets 
def write_critical_sets_sheet(wb, all_issues, found_info=None):
    ws = wb.create_sheet("Critical Sets")
    ws.sheet_view.showGridLines = False

    df = all_issues.filter(
        pl.col("issue_type").is_in(list(CRITICAL_SET_ISSUE_TYPES))
    ).sort(["set_name", "severity", "Q Name"])

    _section_header(ws, 1, "CRITICAL SETS  Structural validation issues", 8)
    next_row = _issues_table(ws, 2, df)

    if found_info:
        next_row += 1
        _section_header(ws, next_row, "QUESTIONS FOUND IN CURRENT QUESTIONNAIRE", 3)
        next_row += 1
        _header_row(ws, next_row, ["Set", "Count found", "Questions"])
        next_row += 1
        for set_name, questions in found_info.items():
            ws.cell(row=next_row, column=1, value=set_name)
            ws.cell(row=next_row, column=2, value=len(questions))
            ws.cell(row=next_row, column=3, value=", ".join(questions) if questions else "none found")
            _data_row(ws, next_row, 3, "info" if questions else "high")
            next_row += 1

    _autofit(ws)


#  Sheet 3: Question Changes 
def write_question_changes_sheet(wb, all_issues):
    """
    All question-level changes sorted highinfo. The 'Type' column shows
    mandatory / mandatory-panel / non-mandatory / optional for each question.
    The 'Field' column is omitted  it is implicit from the issue type.
    """
    ws = wb.create_sheet("Question Changes")
    ws.sheet_view.showGridLines = False

    df = all_issues.filter(
        pl.col("issue_type").is_in([
            "mandatory_column_mismatch", "added_question", "removed_question",
        ])
    )
    df = (
        df.with_columns(
            pl.when(pl.col("severity") == "high"  ).then(pl.lit(0))
            .when(  pl.col("severity") == "medium").then(pl.lit(1))
            .otherwise(pl.lit(2))
            .alias("_s")
        )
        .sort(["_s", "issue_type", "Q Name"])
        .drop("_s")
    )
    for col in ("set_name", "field"):
        if col in df.columns:
            df = df.drop(col)

    _section_header(ws, 1, "QUESTION CHANGES  Presence and mandatory flag (all severities)", 8)
    _issues_table(ws, 2, df)
    _autofit(ws)


#  Sheet 4: Option Changes 
def write_option_changes_sheet(wb, option_changes_view):
    """
    Option-level changes for questions present in both files. The 'Field'
    column shows option_1 / option_2 /  so you know which answer choice is
    affected. The 'Type' column shows the mandatory category of the parent question.
    """
    ws = wb.create_sheet("Option Changes")
    ws.sheet_view.showGridLines = False

    df = option_changes_view
    if "set_name" in df.columns:
        df = df.drop("set_name")
    _section_header(ws, 1, "OPTION CHANGES  Answer options for questions in both files", max(8, len(df.columns)))
    _issues_table(ws, 2, df)
    _autofit(ws)


#  Main export 
def export_validation_report(all_issues, option_changes_view, result_file, rules, found_info=None):
    wb = Workbook()
    wb.remove(wb.active)
    write_summary_sheet(wb, all_issues, rules)
    write_structure_skip_sheet(wb, all_issues)
    write_critical_sets_sheet(wb, all_issues, found_info=found_info)
    write_question_changes_sheet(wb, all_issues)
    write_option_changes_sheet(wb, option_changes_view)
    wb.save(result_file)
    print(f"Report saved to: {result_file}")
    print(f"Sheets: {wb.sheetnames}")


def _resolve_survey_sheet(workbook):
    ws = next((workbook[n] for n in workbook.sheetnames if n.strip().lower() == "survey"), None)
    if ws is None:
        raise KeyError(f"Survey sheet not found. Available sheets: {workbook.sheetnames}")
    return ws


def _sheet_header_map(ws, header_row: int = 3) -> dict[str, int]:
    mapping = {}
    for col in range(1, ws.max_column + 1):
        value = ws.cell(row=header_row, column=col).value
        if value is None:
            continue
        mapping[str(value).strip()] = col
    return mapping


def write_validated_questionnaire(
    source_questionnaire_path: Path,
    output_questionnaire_path: Path,
    questions_df: pl.DataFrame,
) -> None:
    """
    Creates a validated questionnaire workbook by copying the original file and
    writing the converted survey values row-by-row using Q Name as key.
    """
    copyfile(source_questionnaire_path, output_questionnaire_path)

    wb = openpyxl.load_workbook(output_questionnaire_path)
    ws = _resolve_survey_sheet(wb)
    header_map = _sheet_header_map(ws, header_row=3)

    q_col = header_map.get("Q Name")
    if not q_col:
        raise KeyError("Column 'Q Name' not found in survey header row.")

    qname_to_row = {}
    for row_idx in range(4, ws.max_row + 1):
        qname = ws.cell(row=row_idx, column=q_col).value
        key = str(qname).strip() if qname is not None else ""
        if key:
            qname_to_row[key] = row_idx

    writable_cols = [
        col for col in questions_df.columns
        if col not in {"Q Name", "excel_row", "source_file"} and col in header_map
    ]

    for row in questions_df.select(["Q Name"] + writable_cols).to_dicts():
        qname = str(row.get("Q Name") or "").strip()
        target_row = qname_to_row.get(qname)
        if not target_row:
            continue
        for col_name in writable_cols:
            ws.cell(row=target_row, column=header_map[col_name]).value = row.get(col_name)

    wb.save(output_questionnaire_path)
    print(f"Validated questionnaire saved to: {output_questionnaire_path}")


export_validation_report(
    all_issues  = all_issues,
    option_changes_view = option_changes_view,
    result_file = run["report_file"],
    rules       = rules,
    found_info  = _found_info,
)

write_validated_questionnaire(
    source_questionnaire_path = run["questionnaire_path"],
    output_questionnaire_path = run["validated_questionnaire_file"],
    questions_df              = validated_output_questions,
)






Report saved to: C:\Questionnaire_Validation\output\GeoPoll_FAO_FR_DRC_R11_test_geopoll_validation_report.xlsx
Sheets: ['Summary', 'Structure-SkipPattern', 'Critical Sets', 'Question Changes', 'Option Changes']
Validated questionnaire saved to: C:\Questionnaire_Validation\output\GeoPoll_FAO_FR_DRC_R11_test_geopoll_validated_questionnaire.xlsx
